In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════════════╗
# ║ glinker-rs — Knowledgator GLinker → Rust on fbilhaut/orp + ort   (single-cell Colab)      ║
# ║ Steps: deps → write crate → export GLiNER to ONNX → Lean4 proofs → cargo test/build →     ║
# ║        parity vs Python GLiNER → MCC eval (cross-checked with sklearn) → OpenAI server    ║
# ╚══════════════════════════════════════════════════════════════════════════════════════════╝
LINKER_MODEL = "knowledgator/gliner-linker-base-v1.0"   # bi-encoder (L3); large: ...-large-v1.0
NER_MODEL    = None    # e.g. "knowledgator/gliner-bi-base-v2.0" → GLiNER L1; None → gazetteer L1
RERANK_MODEL = None    # e.g. "knowledgator/gliner-linker-rerank-v1.0" → uni-encoder L4
THRESHOLD    = 0.5
PORT         = 8000
WORK         = "/content/glinker_work"

import os, sys, json, subprocess, shutil, time, pathlib, textwrap, glob
def sh(cmd, check=True, env=None, quiet=False):
    print(f"$ {cmd[:160]}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env={**os.environ, **(env or {})})
    if not quiet or r.returncode:
        out = (r.stdout + r.stderr).strip()
        if out: print(out[-4000:])
    if check and r.returncode: raise RuntimeError(f"failed: {cmd}")
    return r.stdout

# ── 0. dependencies ───────────────────────────────────────────────────────────────────────
sh(f"{sys.executable} -m pip install -q gliner==0.2.30 onnx onnxruntime==1.20.1 openai scikit-learn", quiet=True)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", quiet=True)
if not os.path.exists(os.path.expanduser("~/.elan/bin/lean")):
    sh("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.15.0", quiet=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
import onnxruntime as _ort
ORT_DIR = os.path.join(os.path.dirname(_ort.__file__), "capi")
ORT_SO = sorted(glob.glob(os.path.join(ORT_DIR, "libonnxruntime.so*")))[-1]
ENV = {"ORT_LIB_LOCATION": ORT_DIR, "ORT_DYLIB_PATH": ORT_SO, "RUST_BACKTRACE": "0"}
os.environ.update(ENV)

# ── 1. write the crate (sources tested before packaging) ──────────────────────────────────
FILES = {
"Cargo.toml": "[package]\nname = \"glinker-rs\"\nversion = \"0.1.0\"\nedition = \"2021\"\ndescription = \"Rust port of Knowledgator GLinker (zero-shot entity linking) on orp/ort\"\nlicense = \"Apache-2.0\"\n\n[features]\ndefault = [\"load-dynamic\"]\n# Link ONNX Runtime at run time (set ORT_DYLIB_PATH). Works offline / in Colab with pip onnxruntime.\nload-dynamic = [\"orp/load-dynamic\", \"ort/load-dynamic\"]\ncuda = [\"orp/cuda\", \"ort/cuda\"]\ntensorrt = [\"orp/tensorrt\", \"ort/tensorrt\"]\nopenvino = [\"orp/openvino\", \"ort/openvino\"]\n\n[dependencies]\norp = \"=1.0.0\"\nort = \"=2.0.0-rc.9\"\ncomposable = \"1.0\"\nndarray = \"0.16\"\ntokenizers = { version = \"0.21\", default-features = false, features = [\"onig\"] }\nregex = \"1.11\"\nserde = { version = \"1\", features = [\"derive\"] }\nserde_json = \"1\"\naxum = \"0.7\"\ntokio = { version = \"1\", features = [\"rt-multi-thread\", \"macros\", \"net\", \"sync\"] }\n\n[profile.release]\nlto = \"thin\"\n",
"README.md": "# glinker-rs\n\nRust port of [Knowledgator/GLinker](https://github.com/Knowledgator/GLinker) (zero-shot entity linking)\nbuilt on [fbilhaut/orp](https://github.com/fbilhaut/orp) 1.0 + [ort](https://ort.pyke.io) 2.0.0-rc.9.\n\n```\ntext ─L1─▶ mentions ─L2─▶ candidates ─L3 (bi-encoder) / L4 (uni-encoder rerank)─▶ links ─L0─▶ L0Output\n       gazetteer |        dict layer               orp::Pipeline: SpanPipeline          aggregation\n       GLiNER NER        exact + fuzzy             + LabelEncoderPipeline               + stats\n```\n\n| Python (glinker) | Rust |\n|---|---|\n| `l1` spaCy / GLiNER | `l1::L1::{Gliner, Gazetteer, Provided}` |\n| `l2` DictLayer, exact + rapidfuzz | `l2::{DictLayer, L2}`, `fuzz::ratio` (Indel ratio, bit-identical to rapidfuzz) |\n| `l3` GLiNER `predict_with_embeds(input_spans=…)` | `l3::L3` → `gliner::pipeline::SpanPipeline` |\n| `encode_labels` / precomputed embeddings | `gliner::labels::LabelEncoderPipeline`, `Executor::precompute_embeddings` |\n| `l4` reranker with label chunking | `l4::L4` |\n| `l0` aggregation + stats | `l0` |\n| `ProcessorFactory`/`DAGExecutor` | `config::PipelineConfig` (JSON) + `Executor` |\n\nExtra: `metrics` (MCC, candidate- and mention-level), `openai` (`/v1/chat/completions`), `lean/GLinker.lean`.\n\n## Build / run\n```bash\nexport ORT_LIB_LOCATION=$(python -c \"import onnxruntime,os;print(os.path.dirname(onnxruntime.__file__)+'/capi')\")\nexport ORT_DYLIB_PATH=$(ls $ORT_LIB_LOCATION/libonnxruntime.so*|tail -1)     # onnxruntime==1.20.x\ncargo build --release\n./target/release/glinker link  pipeline.json \"TP53 mutations cause cancer.\"\n./target/release/glinker eval  pipeline.json gold.jsonl\n./target/release/glinker serve pipeline.json 0.0.0.0:8000\n```\nModel dirs are produced by the notebook exporter: `model.onnx`, `tokenizer.json`, `gliner_config.json`\n(+ `labels_encoder.onnx`, `labels_tokenizer.json` for bi-encoders).\n\n## Not ported\nspaCy L1 (no Rust equivalent → gazetteer or GLiNER L1), Redis/Elasticsearch/Postgres layers\n(implement `l2::CandidateLayer`), L3 `schema.ranking`, non-whitespace GLiNER word splitters,\ntoken-mode (non-span) GLiNER models.\n",
"lean/GLinker.lean": "/-\n  Formal guarantees for the pure algorithms in glinker-rs (core Lean 4, no Mathlib).\n\n  Rust ↔ Lean correspondence\n  * `gliner::decode::greedy`        ↔ `greedy`, theorem `greedy_pairwise`\n  * `gliner::decode::conflicts`     ↔ `overlap` (flat_ner = true, multi_label = false), `overlap_symm`\n  * threshold filter (`p > t`)      ↔ `keepAbove`, theorem `keepAbove_mono`\n  * `fuzz::lcs` / `fuzz::ratio`     ↔ `lcs`, theorems `lcs_le_left`, `lcs_le_right`, `lcs_comm`\n                                      (so ratio = 2·LCS/(|a|+|b|) ∈ [0,1] and is symmetric)\n  * `encode::span_tensors` mask     ↔ `spanValid`, theorem `spanValid_in_bounds`\n  * `metrics::mcc_binary` numerator ↔ `mccNum`, theorem `mccNum_swap_classes`\n  Scores are modelled as `Nat` (fixed-point); the Rust code compares `f32` with the same `>`.\n-/\n\nnamespace GLinker\n\n/-! ## Greedy span selection is conflict-free -/\n\n/-- `clashes conf x kept` — does `x` conflict with any already-kept element? -/\ndef clashes {α : Type} (conf : α → α → Bool) (x : α) : List α → Bool\n  | [] => false\n  | k :: ks => conf k x || clashes conf x ks\n\n/-- Every element is conflict-free with every element after it. -/\ndef AllPairsOk {α : Type} (conf : α → α → Bool) : List α → Prop\n  | [] => True\n  | k :: ks => clashes conf k ks = false ∧ AllPairsOk conf ks\n\n/-- Greedy pass over spans (already sorted by descending score). Kept spans are consed. -/\ndef greedy {α : Type} (conf : α → α → Bool) : List α → List α → List α\n  | kept, [] => kept\n  | kept, x :: xs => if clashes conf x kept then greedy conf kept xs else greedy conf (x :: kept) xs\n\ntheorem greedy_pairwise {α : Type} (conf : α → α → Bool) :\n    ∀ (xs kept : List α), AllPairsOk conf kept → AllPairsOk conf (greedy conf kept xs)\n  | [], kept, h => h\n  | x :: xs, kept, h => by\n    unfold greedy\n    split\n    · exact greedy_pairwise conf xs kept h\n    · next hc =>\n      have hk : AllPairsOk conf (x :: kept) := by\n        simp only [AllPairsOk]\n        exact And.intro (by simpa using hc) h\n      exact greedy_pairwise conf xs (x :: kept) hk\n\n/-- Starting from nothing, greedy output is pairwise conflict-free. -/\ntheorem greedy_sound {α : Type} (conf : α → α → Bool) (xs : List α) :\n    AllPairsOk conf (greedy conf [] xs) :=\n  greedy_pairwise conf xs [] trivial\n\n/-- Greedy never invents spans: everything kept came from the input. -/\ntheorem greedy_subset {α : Type} (conf : α → α → Bool) :\n    ∀ (xs kept : List α) (y : α), y ∈ greedy conf kept xs → y ∈ kept ∨ y ∈ xs\n  | [], kept, y, h => Or.inl h\n  | x :: xs, kept, y, h => by\n    unfold greedy at h\n    split at h\n    · cases greedy_subset conf xs kept y h with\n      | inl hk => exact Or.inl hk\n      | inr hx => exact Or.inr (List.Mem.tail _ hx)\n    · cases greedy_subset conf xs (x :: kept) y h with\n      | inl hk =>\n        cases hk with\n        | head => exact Or.inr (List.Mem.head _)\n        | tail _ hk' => exact Or.inl hk'\n      | inr hx => exact Or.inr (List.Mem.tail _ hx)\n\n/-! ## The flat-NER conflict relation is symmetric -/\n\nstructure Span where\n  s : Nat\n  e : Nat\n  deriving Repr\n\n/-- GLiNER `has_overlapping` with `multi_label = false`: spans conflict unless disjoint. -/\ndef overlap (a b : Span) : Bool :=\n  !(decide (b.e < a.s) || decide (a.e < b.s))\n\ntheorem overlap_symm (a b : Span) : overlap a b = overlap b a := by\n  unfold overlap\n  rw [Bool.or_comm]\n\n/-! ## Threshold monotonicity: raising the threshold only removes predictions -/\n\ndef keepAbove (t : Nat) : List Nat → List Nat\n  | [] => []\n  | x :: xs => if t < x then x :: keepAbove t xs else keepAbove t xs\n\ntheorem keepAbove_mono {t1 t2 : Nat} (h : t1 ≤ t2) :\n    ∀ (l : List Nat) (x : Nat), x ∈ keepAbove t2 l → x ∈ keepAbove t1 l\n  | [], x, hx => by simp [keepAbove] at hx\n  | y :: ys, x, hx => by\n    unfold keepAbove at hx ⊢\n    by_cases h2 : t2 < y\n    · rw [if_pos h2] at hx\n      rw [if_pos (Nat.lt_of_le_of_lt h h2)]\n      cases hx with\n      | head => exact List.Mem.head _\n      | tail _ hm => exact List.Mem.tail _ (keepAbove_mono h ys x hm)\n    · rw [if_neg h2] at hx\n      have ih := keepAbove_mono h ys x hx\n      by_cases h1 : t1 < y\n      · rw [if_pos h1]; exact List.Mem.tail _ ih\n      · rw [if_neg h1]; exact ih\n\n/-! ## LCS (rapidfuzz `ratio` = 2·LCS/(|a|+|b|)) is bounded and symmetric -/\n\ndef mx (a b : Nat) : Nat := if a ≤ b then b else a\n\ntheorem mx_le {a b c : Nat} (h1 : a ≤ c) (h2 : b ≤ c) : mx a b ≤ c := by\n  unfold mx; split <;> assumption\n\ntheorem mx_comm (a b : Nat) : mx a b = mx b a := by\n  unfold mx; split <;> split <;> omega\n\ndef lcs : List Nat → List Nat → Nat\n  | [], _ => 0\n  | _ :: _, [] => 0\n  | a :: as, b :: bs => if a = b then lcs as bs + 1 else mx (lcs as (b :: bs)) (lcs (a :: as) bs)\n\ntheorem lcs_le_left : ∀ (xs ys : List Nat), lcs xs ys ≤ xs.length\n  | [], _ => by simp [lcs]\n  | _ :: _, [] => by simp [lcs]\n  | a :: as, b :: bs => by\n    simp only [lcs]\n    split\n    · have := lcs_le_left as bs\n      simp only [List.length_cons]; omega\n    · apply mx_le\n      · have := lcs_le_left as (b :: bs)\n        simp only [List.length_cons]; omega\n      · exact lcs_le_left (a :: as) bs\n\ntheorem lcs_comm : ∀ (xs ys : List Nat), lcs xs ys = lcs ys xs\n  | [], [] => rfl\n  | [], _ :: _ => by simp [lcs]\n  | _ :: _, [] => by simp [lcs]\n  | a :: as, b :: bs => by\n    simp only [lcs]\n    by_cases h : a = b\n    · rw [if_pos h, if_pos h.symm, lcs_comm as bs]\n    · have h' : ¬ b = a := fun e => h e.symm\n      rw [if_neg h, if_neg h', lcs_comm as (b :: bs), lcs_comm (a :: as) bs, mx_comm]\n\ntheorem lcs_le_right (xs ys : List Nat) : lcs xs ys ≤ ys.length := by\n  rw [lcs_comm]; exact lcs_le_left ys xs\n\n/-- 2·LCS ≤ |a|+|b|, i.e. `ratio ≤ 1` (and `ratio ≥ 0` is trivial over Nat). -/\ntheorem ratio_le_one (xs ys : List Nat) : 2 * lcs xs ys ≤ xs.length + ys.length := by\n  have := lcs_le_left xs ys\n  have := lcs_le_right xs ys\n  omega\n\n/-! ## Span enumeration stays inside the text -/\n\n/-- `span_tensors` marks (i, i+w) valid iff i + w < len. -/\ndef spanValid (len i w : Nat) : Bool := decide (i + w < len)\n\ntheorem spanValid_in_bounds (len i w : Nat) (h : spanValid len i w = true) :\n    i < len ∧ i + w < len := by\n  unfold spanValid at h\n  have := of_decide_eq_true h\n  omega\n\n/-! ## MCC numerator is invariant under swapping the positive class -/\n\ndef mccNum (tp tn fp fn : Int) : Int := tp * tn - fp * fn\n\n/-- Relabelling positives ↔ negatives maps (tp,tn,fp,fn) ↦ (tn,tp,fn,fp). -/\ntheorem mccNum_swap_classes (tp tn fp fn : Int) : mccNum tp tn fp fn = mccNum tn tp fn fp := by\n  unfold mccNum\n  rw [Int.mul_comm tp tn, Int.mul_comm fp fn]\n\n/-- Perfect prediction (fp = fn = 0) gives a non-negative numerator. -/\ntheorem mccNum_perfect_nonneg (tp tn : Nat) : 0 ≤ mccNum tp tn 0 0 := by\n  unfold mccNum\n  rw [Int.mul_zero, Int.sub_zero]\n  exact Int.mul_nonneg (by omega) (by omega)\n\nend GLinker\n",
"src/bin/glinker.rs": "//! CLI:\n//!   glinker link  <config.json> \"text\" [\"text\" ...]   → L0 JSON on stdout\n//!   glinker eval  <config.json> <gold.jsonl>          → MCC report JSON\n//!   glinker serve <config.json> [addr=0.0.0.0:8000]   → OpenAI-compatible server\n//!\n//! ONNX Runtime is loaded from `ORT_DYLIB_PATH` (feature `load-dynamic`, the default).\n\nuse std::sync::{Arc, Mutex};\nuse glinker_rs::metrics::{evaluate, GoldDoc};\nuse glinker_rs::openai::{serve, AppState};\nuse glinker_rs::Executor;\n\nfn usage() -> ! {\n    eprintln!(\"usage: glinker <link|eval|serve> <config.json> [...]\");\n    std::process::exit(2)\n}\n\nfn main() -> glinker_rs::Result<()> {\n    let args: Vec<String> = std::env::args().collect();\n    if args.len() < 3 {\n        usage();\n    }\n    let mut ex = Executor::from_config_file(&args[2])?;\n    match args[1].as_str() {\n        \"link\" => {\n            let texts: Vec<String> = args[3..].to_vec();\n            let out = ex.execute(&texts)?;\n            println!(\"{}\", serde_json::to_string_pretty(&out)?);\n        }\n        \"eval\" => {\n            let path = args.get(3).unwrap_or_else(|| usage());\n            let gold: Vec<GoldDoc> = std::fs::read_to_string(path)?\n                .lines()\n                .filter(|l| !l.trim().is_empty())\n                .map(serde_json::from_str)\n                .collect::<Result<_, _>>()?;\n            let texts: Vec<String> = gold.iter().map(|g| g.text.clone()).collect();\n            let out = ex.execute(&texts)?;\n            let report = evaluate(&gold, &out.entities);\n            println!(\"{}\", serde_json::to_string_pretty(&report)?);\n        }\n        \"serve\" => {\n            let addr = args.get(3).cloned().unwrap_or_else(|| \"0.0.0.0:8000\".into());\n            let name = if ex.name.is_empty() { \"glinker-rs\".to_string() } else { ex.name.clone() };\n            let state = AppState { executor: Arc::new(Mutex::new(ex)), model_name: name };\n            eprintln!(\"glinker-rs listening on http://{addr}/v1/chat/completions\");\n            tokio::runtime::Runtime::new()?.block_on(serve(state, &addr))?;\n        }\n        _ => usage(),\n    }\n    Ok(())\n}\n",
"src/config.rs": "//! JSON pipeline config — same shape as GLinker's `ConfigBuilder` layers.\n//!\n//! ```json\n//! {\n//!   \"name\": \"biomed\",\n//!   \"template\": \"{label}: {description}\",\n//!   \"entities\": \"data/entities.jsonl\",\n//!   \"precompute_embeddings\": true,\n//!   \"runtime\": { \"threads\": 4 },\n//!   \"l1\": { \"type\": \"gliner\", \"model_dir\": \"models/ner\", \"labels\": [\"gene\", \"disease\"], \"threshold\": 0.3 },\n//!   \"l2\": { \"max_candidates\": 30, \"search_mode\": [\"exact\", \"fuzzy\"] },\n//!   \"l3\": { \"model_dir\": \"models/linker\", \"model_id\": \"knowledgator/gliner-linker-base-v1.0\", \"threshold\": 0.5 },\n//!   \"l4\": { \"model_dir\": \"models/rerank\", \"max_labels\": 20 },\n//!   \"l0\": { \"strict_matching\": true, \"position_tolerance\": 2 }\n//! }\n//! ```\n//! `\"l1\": {\"type\": \"none\"}` gives the Python *simple* pipeline (all entities are candidates,\n//! loose L0 matching).\n\nuse serde::Deserialize;\nuse crate::l2::{FuzzyConfig, SearchMode};\n\n#[derive(Clone, Debug, Deserialize)]\npub struct PipelineConfig {\n    #[serde(default)]\n    pub name: String,\n    #[serde(default = \"d_tpl\")]\n    pub template: String,\n    #[serde(default)]\n    pub entities: Option<String>,\n    #[serde(default)]\n    pub precompute_embeddings: bool,\n    #[serde(default)]\n    pub runtime: RuntimeConfig,\n    pub l1: L1Config,\n    #[serde(default)]\n    pub l2: L2Config,\n    pub l3: ModelConfig,\n    #[serde(default)]\n    pub l4: Option<ModelConfig>,\n    #[serde(default)]\n    pub l0: L0Config,\n}\nfn d_tpl() -> String { \"{label}\".into() }\n\n#[derive(Clone, Debug, Deserialize)]\npub struct RuntimeConfig {\n    #[serde(default = \"d_threads\")]\n    pub threads: usize,\n}\nfn d_threads() -> usize { 4 }\nimpl Default for RuntimeConfig {\n    fn default() -> Self { Self { threads: 4 } }\n}\n\n#[derive(Clone, Debug, Deserialize)]\n#[serde(tag = \"type\", rename_all = \"lowercase\")]\npub enum L1Config {\n    None,\n    Gazetteer {\n        #[serde(default = \"d_min_len\")]\n        min_entity_length: usize,\n    },\n    Gliner {\n        model_dir: String,\n        labels: Vec<String>,\n        #[serde(default = \"d_l1_thr\")]\n        threshold: f32,\n        #[serde(default = \"d_min_len\")]\n        min_entity_length: usize,\n        #[serde(default = \"d_label_tokens\")]\n        label_max_tokens: usize,\n    },\n}\nfn d_min_len() -> usize { 2 }\nfn d_l1_thr() -> f32 { 0.3 }\n\n#[derive(Clone, Debug, Deserialize)]\npub struct L2Config {\n    /// `\"search\"` (default): per-mention retrieval. `\"all\"`: whole database is every text's\n    /// candidate set (Python `mentions=None`), while L1 spans still constrain L3.\n    #[serde(default = \"d_l2_mode\")]\n    pub mode: String,\n    #[serde(default = \"d_max_c\")]\n    pub max_candidates: usize,\n    #[serde(default)]\n    pub min_popularity: i64,\n    #[serde(default = \"d_modes\")]\n    pub search_mode: Vec<SearchMode>,\n    #[serde(default)]\n    pub fuzzy: FuzzyConfig,\n}\nfn d_max_c() -> usize { 30 }\nfn d_l2_mode() -> String { \"search\".into() }\nfn d_modes() -> Vec<SearchMode> { vec![SearchMode::Exact] }\nimpl Default for L2Config {\n    fn default() -> Self { Self { mode: d_l2_mode(), max_candidates: 30, min_popularity: 0, search_mode: d_modes(), fuzzy: FuzzyConfig::default() } }\n}\n\n#[derive(Clone, Debug, Deserialize)]\npub struct ModelConfig {\n    pub model_dir: String,\n    #[serde(default)]\n    pub model_id: String,\n    #[serde(default = \"d_thr\")]\n    pub threshold: f32,\n    #[serde(default = \"d_true\")]\n    pub flat_ner: bool,\n    #[serde(default)]\n    pub multi_label: bool,\n    #[serde(default = \"d_true\")]\n    pub use_precomputed_embeddings: bool,\n    #[serde(default)]\n    pub cache_embeddings: bool,\n    #[serde(default = \"d_label_tokens\")]\n    pub label_max_tokens: usize,\n    /// L4 only\n    #[serde(default = \"d_max_labels\")]\n    pub max_labels: usize,\n}\nfn d_thr() -> f32 { 0.5 }\nfn d_true() -> bool { true }\nfn d_label_tokens() -> usize { 128 }\nfn d_max_labels() -> usize { 20 }\n\n#[derive(Clone, Debug, Deserialize)]\npub struct L0Config {\n    #[serde(default)]\n    pub min_confidence: f32,\n    #[serde(default = \"d_true\")]\n    pub include_unlinked: bool,\n    #[serde(default)]\n    pub strict_matching: Option<bool>,\n    #[serde(default = \"d_tol\")]\n    pub position_tolerance: usize,\n}\nfn d_tol() -> usize { 2 }\nimpl Default for L0Config {\n    fn default() -> Self { Self { min_confidence: 0.0, include_unlinked: true, strict_matching: None, position_tolerance: 2 } }\n}\n",
"src/executor.rs": "//! DAG executor: L1 → L2 → L3 (or L4) → L0 (Python `DAGExecutor` for the built-in topologies).\n//! When L4 is configured its output feeds L0, as in Python; L3 is then skipped because Python\n//! discards its result.\n\nuse std::path::Path;\nuse orp::params::RuntimeParameters;\n\nuse crate::config::*;\nuse crate::gliner::engine::{LabelEncoder, SpanModel};\nuse crate::gliner::GlinerParams;\nuse crate::l0::{self, L0Options};\nuse crate::l1::{L1Options, L1};\nuse crate::l2::{DictLayer, L2Options, L2};\nuse crate::l3::L3;\nuse crate::l4::L4;\nuse crate::types::*;\nuse crate::Result;\n\npub struct Executor {\n    pub name: String,\n    pub l1: Option<L1>,\n    pub l2: L2,\n    pub l3: L3,\n    pub l4: Option<L4>,\n    pub l0: L0Options,\n    pub template: String,\n    pub cache_embeddings: bool,\n    /// L2 `\"mode\": \"all\"`\n    pub all_candidates: bool,\n}\n\nfn rt(c: &RuntimeConfig) -> RuntimeParameters {\n    let p = RuntimeParameters::default().with_threads(c.threads);\n    #[cfg(feature = \"cuda\")]\n    let p = p.with_execution_providers([ort::execution_providers::CUDAExecutionProvider::default().build()]);\n    p\n}\n\nfn params(m: &ModelConfig) -> GlinerParams {\n    GlinerParams { threshold: m.threshold, flat_ner: m.flat_ner, multi_label: m.multi_label, ..Default::default() }\n}\n\nimpl Executor {\n    pub fn from_config_file(path: impl AsRef<Path>) -> Result<Self> {\n        let path = path.as_ref();\n        let cfg: PipelineConfig = serde_json::from_str(&std::fs::read_to_string(path)?)?;\n        let base = path.parent().unwrap_or(Path::new(\".\"));\n        Self::from_config(cfg, base)\n    }\n\n    /// Relative paths in `cfg` are resolved against `base`.\n    pub fn from_config(cfg: PipelineConfig, base: &Path) -> Result<Self> {\n        let res = |p: &str| base.join(p);\n        let mut dict = DictLayer::new(0, cfg.l2.search_mode.clone(), cfg.l2.fuzzy.clone());\n        if let Some(e) = &cfg.entities {\n            dict.load_file(res(e).to_str().ok_or(\"bad path\")?)?;\n        }\n        let entities_snapshot = {\n            use crate::l2::CandidateLayer;\n            dict.all()\n        };\n        let l2 = L2::new(vec![Box::new(dict)], L2Options { max_candidates: cfg.l2.max_candidates, min_popularity: cfg.l2.min_popularity });\n\n        let l1 = match &cfg.l1 {\n            L1Config::None => None,\n            L1Config::Gazetteer { min_entity_length } => {\n                Some(L1::gazetteer(&entities_snapshot, L1Options { min_entity_length: *min_entity_length, ..Default::default() }))\n            }\n            L1Config::Gliner { model_dir, labels, threshold, min_entity_length, label_max_tokens } => {\n                let p = GlinerParams { threshold: *threshold, ..Default::default() };\n                let m = SpanModel::load(res(model_dir), model_dir, p, rt(&cfg.runtime))?;\n                let enc = if m.is_bi_encoder() { Some(LabelEncoder::load(res(model_dir), *label_max_tokens, rt(&cfg.runtime))?) } else { None };\n                Some(L1::gliner(m, labels.clone(), enc.as_ref(), L1Options { min_entity_length: *min_entity_length, ..Default::default() })?)\n            }\n        };\n\n        let m3 = &cfg.l3;\n        let model_id = if m3.model_id.is_empty() { m3.model_dir.clone() } else { m3.model_id.clone() };\n        let span = SpanModel::load(res(&m3.model_dir), &model_id, params(m3), rt(&cfg.runtime))?;\n        let encoder = if span.is_bi_encoder() { Some(LabelEncoder::load(res(&m3.model_dir), m3.label_max_tokens, rt(&cfg.runtime))?) } else { None };\n        let l3 = L3 { model: span, encoder, template: cfg.template.clone(), use_precomputed_embeddings: m3.use_precomputed_embeddings };\n\n        let l4 = match &cfg.l4 {\n            Some(m4) => {\n                let id = if m4.model_id.is_empty() { m4.model_dir.clone() } else { m4.model_id.clone() };\n                let model = SpanModel::load(res(&m4.model_dir), &id, params(m4), rt(&cfg.runtime))?;\n                if model.is_bi_encoder() {\n                    return Err(\"L4 reranker must be a uni-encoder GLiNER\".into());\n                }\n                Some(L4 { model, template: cfg.template.clone(), max_labels: m4.max_labels })\n            }\n            None => None,\n        };\n\n        let simple = l1.is_none();\n        let l0 = L0Options {\n            min_confidence: cfg.l0.min_confidence,\n            include_unlinked: cfg.l0.include_unlinked,\n            strict_matching: cfg.l0.strict_matching.unwrap_or(!simple),\n            position_tolerance: cfg.l0.position_tolerance,\n        };\n        if cfg.l2.mode != \"search\" && cfg.l2.mode != \"all\" {\n            return Err(format!(\"unknown l2.mode {:?}\", cfg.l2.mode).into());\n        }\n        let all_candidates = cfg.l2.mode == \"all\";\n        let mut ex = Executor { name: cfg.name, l1, l2, l3, l4, l0, template: cfg.template, cache_embeddings: m3.cache_embeddings, all_candidates };\n        if cfg.precompute_embeddings {\n            ex.precompute_embeddings()?;\n        }\n        Ok(ex)\n    }\n\n    pub fn load_entities(&mut self, entities: Vec<DatabaseRecord>) -> Result<()> {\n        let mut d = DictLayer::new(0, vec![crate::l2::SearchMode::Exact], Default::default());\n        d.load_bulk(entities, true);\n        self.l2.layers.push(Box::new(d));\n        Ok(())\n    }\n\n    pub fn precompute_embeddings(&mut self) -> Result<usize> {\n        let all = self.l2.all();\n        let new = self.l3.precompute(&all)?;\n        let (ids, embs): (Vec<_>, Vec<_>) = new.into_iter().unzip();\n        let id = self.l3.model.model_id.clone();\n        Ok(self.l2.update_embeddings(&ids, &embs, &id))\n    }\n\n    /// Full pipeline on raw texts.\n    pub fn execute(&mut self, texts: &[String]) -> Result<L0Output> {\n        let mentions = match &self.l1 {\n            Some(l1) => Some(l1.extract(texts)?),\n            None => None,\n        };\n        self.run(texts, mentions)\n    }\n\n    /// Pipeline with caller-supplied mentions (skips L1).\n    pub fn link_with_mentions(&mut self, texts: &[String], mentions: Vec<Vec<Mention>>) -> Result<L0Output> {\n        self.run(texts, Some(mentions))\n    }\n\n    fn run(&mut self, texts: &[String], mentions: Option<Vec<Vec<Mention>>>) -> Result<L0Output> {\n        // L2\n        let (per_mention, per_text, shared): (Vec<Vec<Vec<DatabaseRecord>>>, Vec<Vec<DatabaseRecord>>, bool) = match &mentions {\n            None => (vec![vec![]; texts.len()], vec![self.l2.all()], true),\n            Some(ms) => {\n                let pm: Vec<Vec<Vec<DatabaseRecord>>> = ms.iter().map(|v| v.iter().map(|m| self.l2.search(&m.text)).collect()).collect();\n                if self.all_candidates {\n                    (pm, vec![self.l2.all()], true)\n                } else {\n                    let pt = pm.iter().map(|v| v.iter().flatten().cloned().collect()).collect();\n                    (pm, pt, false)\n                }\n            }\n        };\n        // L3 / L4\n        let links = if let Some(l4) = &self.l4 {\n            l4.rerank(texts, &per_text, mentions.as_deref(), shared)?\n        } else {\n            let (links, fresh) = self.l3.link(texts, &per_text, mentions.as_deref(), shared)?;\n            if self.cache_embeddings && !fresh.is_empty() {\n                let (ids, embs): (Vec<_>, Vec<_>) = fresh.into_iter().unzip();\n                let id = self.l3.model.model_id.clone();\n                self.l2.update_embeddings(&ids, &embs, &id);\n            }\n            links\n        };\n        // L0\n        let empty: Vec<Mention> = vec![];\n        let entities: Vec<Vec<L0Entity>> = (0..texts.len())\n            .map(|i| {\n                let ms = mentions.as_ref().map(|m| m[i].as_slice()).unwrap_or(&empty);\n                let tc = if shared { &per_text[0] } else { &per_text[i] };\n                l0::aggregate_text(ms, &per_mention[i], tc, &links[i], &self.template, &self.l0)\n            })\n            .collect();\n        let stats = l0::stats(&entities);\n        Ok(L0Output { entities, stats })\n    }\n}\n",
"src/fuzz.rs": "//! Port of `rapidfuzz.fuzz.ratio` (normalized Indel similarity), used by the L2 dict layer.\n//!\n//! `ratio(a, b) = 1 - indel(a, b) / (|a| + |b|)` where `indel = |a| + |b| - 2·LCS(a, b)`,\n//! i.e. `ratio = 2·LCS / (|a| + |b|)`; both empty → 1.0.\n//! Lean4 (`lean/GLinker.lean`): `lcs_le_min` gives 0 ≤ ratio ≤ 1, and `lcs_comm` gives symmetry.\n\n/// Longest common subsequence length over chars, O(|a|·|b|) time, O(|b|) memory.\npub fn lcs(a: &[char], b: &[char]) -> usize {\n    let mut prev = vec![0usize; b.len() + 1];\n    let mut cur = vec![0usize; b.len() + 1];\n    for &x in a {\n        for (j, &y) in b.iter().enumerate() {\n            cur[j + 1] = if x == y { prev[j] + 1 } else { prev[j + 1].max(cur[j]) };\n        }\n        std::mem::swap(&mut prev, &mut cur);\n    }\n    prev[b.len()]\n}\n\n/// `rapidfuzz.fuzz.ratio(a, b) / 100`.\npub fn ratio(a: &str, b: &str) -> f64 {\n    let a: Vec<char> = a.chars().collect();\n    let b: Vec<char> = b.chars().collect();\n    let total = a.len() + b.len();\n    if total == 0 {\n        return 1.0;\n    }\n    2.0 * lcs(&a, &b) as f64 / total as f64\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    #[test]\n    fn matches_rapidfuzz() {\n        // rapidfuzz.fuzz.ratio(\"this is a test\", \"this is a test!\") == 96.55172413793103\n        assert!((ratio(\"this is a test\", \"this is a test!\") - 0.9655172413793103).abs() < 1e-12);\n        // fuzz.ratio(\"insulin\", \"insuline\") == 93.33333333333333\n        assert!((ratio(\"insulin\", \"insuline\") - 0.9333333333333333).abs() < 1e-12);\n        assert_eq!(ratio(\"\", \"\"), 1.0);\n        assert_eq!(ratio(\"abc\", \"\"), 0.0);\n        assert_eq!(ratio(\"ab\", \"ba\"), ratio(\"ba\", \"ab\"));\n    }\n}\n",
"src/gliner/decode.rs": "//! Pure span decoding — port of `gliner.decoding.SpanDecoder` (`_decode_batch` + `greedy_search`).\n//! Kept free of ONNX types so it is unit-testable and mirrors the Lean model in `lean/GLinker.lean`.\n\nuse std::collections::HashSet;\nuse ndarray::ArrayView3;\n\n#[derive(Clone, Debug, PartialEq)]\npub struct RawSpan {\n    /// inclusive word indices\n    pub start: usize,\n    pub end: usize,\n    pub class: usize,\n    pub score: f32,\n    /// (class index, prob), descending\n    pub class_probs: Vec<(usize, f32)>,\n}\n\n#[inline]\npub fn sigmoid(x: f32) -> f32 {\n    1.0 / (1.0 + (-x).exp())\n}\n\n/// Decode one item. `logits` has shape (L, K, C) (words, widths, classes).\n/// - `filter`: GLinker `input_spans` as word spans (start, end) inclusive; `None` = all spans.\n/// - keeps `prob > threshold` (strict, as in Python), and `start + width + 1 <= num_words`.\npub fn decode_item(\n    logits: ArrayView3<f32>,\n    num_words: usize,\n    filter: Option<&HashSet<(usize, usize)>>,\n    threshold: f32,\n    top_k: usize,\n) -> Vec<RawSpan> {\n    let (l, k, c) = logits.dim();\n    let mut out = Vec::new();\n    for s in 0..l.min(num_words) {\n        for w in 0..k {\n            let e = s + w;\n            if e + 1 > num_words {\n                break;\n            }\n            if let Some(f) = filter {\n                if !f.contains(&(s, e)) {\n                    continue;\n                }\n            }\n            let mut probs: Option<Vec<(usize, f32)>> = None;\n            for cls in 0..c {\n                let p = sigmoid(logits[[s, w, cls]]);\n                if p > threshold {\n                    let cp = probs\n                        .get_or_insert_with(|| {\n                            let mut v: Vec<(usize, f32)> = (0..c).map(|j| (j, sigmoid(logits[[s, w, j]]))).collect();\n                            v.sort_by(|a, b| b.1.total_cmp(&a.1));\n                            v.truncate(top_k);\n                            v\n                        })\n                        .clone();\n                    out.push(RawSpan { start: s, end: e, class: cls, score: p, class_probs: cp });\n                }\n            }\n        }\n    }\n    out\n}\n\n/// `has_overlapping` (flat) / `has_overlapping_nested` (nested), with the multi-label same-span rule.\npub fn conflicts(a: &RawSpan, b: &RawSpan, flat_ner: bool, multi_label: bool) -> bool {\n    if a.start == b.start && a.end == b.end {\n        return !multi_label;\n    }\n    let disjoint = a.start > b.end || b.start > a.end;\n    if flat_ner {\n        !disjoint\n    } else {\n        let nested = (a.start <= b.start && a.end >= b.end) || (b.start <= a.start && b.end >= a.end);\n        !(disjoint || nested)\n    }\n}\n\n/// Greedy search: highest score first, keep a span iff it conflicts with no kept span;\n/// result ordered by (start, end). Proven pairwise conflict-free in Lean (`greedy_pairwise`).\npub fn greedy(mut spans: Vec<RawSpan>, flat_ner: bool, multi_label: bool) -> Vec<RawSpan> {\n    spans.sort_by(|a, b| b.score.total_cmp(&a.score)); // stable, like Python `sorted`\n    let mut kept: Vec<RawSpan> = Vec::new();\n    for s in spans {\n        if !kept.iter().any(|k| conflicts(k, &s, flat_ner, multi_label)) {\n            kept.push(s);\n        }\n    }\n    kept.sort_by_key(|s| (s.start, s.end));\n    kept\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    use ndarray::Array3;\n\n    fn logit(p: f32) -> f32 { (p / (1.0 - p)).ln() }\n\n    #[test]\n    fn filter_threshold_greedy() {\n        // 3 words, K=2, C=2\n        let mut a = Array3::from_elem((3, 2, 2), -20.0f32);\n        a[[0, 0, 0]] = logit(0.9); // (0,0) c0\n        a[[0, 1, 1]] = logit(0.8); // (0,1) c1 overlaps (0,0)\n        a[[2, 0, 1]] = logit(0.7); // (2,2) c1\n        a[[2, 1, 0]] = logit(0.99); // (2,3) invalid: beyond num_words\n        let all = decode_item(a.view(), 3, None, 0.5, 5);\n        assert_eq!(all.len(), 3);\n        let g = greedy(all, true, false);\n        assert_eq!(g.iter().map(|s| (s.start, s.end, s.class)).collect::<Vec<_>>(), vec![(0, 0, 0), (2, 2, 1)]);\n        // input-span constraint keeps only (0,1)\n        let f: HashSet<_> = [(0usize, 1usize)].into_iter().collect();\n        let c = decode_item(a.view(), 3, Some(&f), 0.5, 5);\n        assert_eq!(c.len(), 1);\n        assert_eq!((c[0].start, c[0].end, c[0].class), (0, 1, 1));\n        assert_eq!(c[0].class_probs[0].0, 1);\n    }\n\n    #[test]\n    fn nested_mode_allows_nesting() {\n        let mk = |s, e, sc| RawSpan { start: s, end: e, class: 0, score: sc, class_probs: vec![] };\n        let g = greedy(vec![mk(0, 3, 0.9), mk(1, 2, 0.8), mk(2, 5, 0.7)], false, false);\n        assert_eq!(g.len(), 2); // (2,5) crosses (0,3)\n    }\n}\n",
"src/gliner/encode.rs": "//! Sub-word encoding — port of GLiNER `tokenize_inputs` + `prepare_word_mask` (first-subtoken\n//! pooling) + `prepare_span_idx`. Special tokens come from `tokenizer.json`'s post-processor,\n//! so DeBERTa (`[CLS]=1`) and ModernBERT/Ettin (`[CLS]=50281`) backbones both work.\n\nuse ndarray::{Array2, Array3};\nuse tokenizers::Tokenizer;\nuse crate::text::Word;\nuse super::Pooling;\nuse crate::Result;\n\npub struct Encoded {\n    pub input_ids: Array2<i64>,\n    pub attention_mask: Array2<i64>,\n    pub words_mask: Array2<i64>,\n    pub text_lengths: Array2<i64>,\n    /// max words over the batch (the `L` dimension of logits)\n    pub num_words: usize,\n}\n\n/// Load a tokenizer and strip any truncation/padding baked into `tokenizer.json`.\npub fn load_tokenizer<P: AsRef<std::path::Path>>(p: P) -> Result<Tokenizer> {\n    let mut t = Tokenizer::from_file(p)?;\n    t.with_truncation(None)?;\n    t.with_padding(None);\n    Ok(t)\n}\n\n/// Encode `prompt ++ words` for each text. `prompt` is empty for bi-encoders.\npub fn encode(tokenizer: &Tokenizer, prompt: &[String], texts: &[Vec<Word>], pooling: Pooling) -> Result<Encoded> {\n    let skip = prompt.len();\n    let mut rows: Vec<(Vec<i64>, Vec<i64>)> = Vec::with_capacity(texts.len());\n    for words in texts {\n        let seq: Vec<&str> = prompt.iter().map(String::as_str).chain(words.iter().map(|w| w.text.as_str())).collect();\n        let enc = tokenizer.encode(seq.as_slice(), true)?;\n        let ids: Vec<i64> = enc.get_ids().iter().map(|&x| x as i64).collect();\n        let wids = enc.get_word_ids();\n        let mut mask = Vec::with_capacity(ids.len());\n        let mut prev: Option<u32> = None;\n        for (ti, wid) in wids.iter().enumerate() {\n            match *wid {\n                None => mask.push(0),\n                Some(w) => {\n                    let first = Some(w) != prev;\n                    let last = wids.get(ti + 1).copied().flatten() != Some(w);\n                    let pick = match pooling { Pooling::First => first, Pooling::Last => last, Pooling::All => true };\n                    let w = w as usize;\n                    mask.push(if w >= skip && pick { (w - skip + 1) as i64 } else { 0 });\n                }\n            }\n            prev = *wid;\n        }\n        rows.push((ids, mask));\n    }\n    let b = texts.len();\n    let t = rows.iter().map(|r| r.0.len()).max().unwrap_or(0);\n    let mut input_ids = Array2::<i64>::zeros((b, t));\n    let mut attention_mask = Array2::<i64>::zeros((b, t));\n    let mut words_mask = Array2::<i64>::zeros((b, t));\n    for (i, (ids, m)) in rows.iter().enumerate() {\n        for j in 0..ids.len() {\n            input_ids[[i, j]] = ids[j];\n            attention_mask[[i, j]] = 1;\n            words_mask[[i, j]] = m[j];\n        }\n    }\n    let mut text_lengths = Array2::<i64>::zeros((b, 1));\n    for (i, w) in texts.iter().enumerate() {\n        text_lengths[[i, 0]] = w.len() as i64;\n    }\n    let num_words = texts.iter().map(Vec::len).max().unwrap_or(0);\n    Ok(Encoded { input_ids, attention_mask, words_mask, text_lengths, num_words })\n}\n\n/// `prepare_span_idx` for a padded batch: shape (B, L·K, 2) and mask (B, L·K).\n/// Span `(i, i+w)` is valid iff `i + w < len_b`. Lean: `span_count` proves the valid count.\npub fn span_tensors(lens: &[usize], num_words: usize, max_width: usize) -> (Array3<i64>, Array2<bool>) {\n    let b = lens.len();\n    let n = num_words * max_width;\n    let mut idx = Array3::<i64>::zeros((b, n, 2));\n    let mut mask = Array2::<bool>::from_elem((b, n), false);\n    for (bi, &len) in lens.iter().enumerate() {\n        for i in 0..num_words {\n            for w in 0..max_width {\n                let j = i * max_width + w;\n                if i + w < len {\n                    idx[[bi, j, 0]] = i as i64;\n                    idx[[bi, j, 1]] = (i + w) as i64;\n                    mask[[bi, j]] = true;\n                }\n            }\n        }\n    }\n    (idx, mask)\n}\n\n/// GLiNER uni-encoder prompt: `[ent, label]* sep`.\npub fn uni_prompt(labels: &[String], ent_token: &str, sep_token: &str) -> Vec<String> {\n    let mut p = Vec::with_capacity(labels.len() * 2 + 1);\n    for l in labels {\n        p.push(ent_token.to_string());\n        if !l.is_empty() {\n            p.push(l.clone());\n        }\n    }\n    p.push(sep_token.to_string());\n    p\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    #[test]\n    fn span_mask_count() {\n        let (_, m) = span_tensors(&[4, 2], 4, 3);\n        // len 4, K 3: 3+3+2+1 = 9 ; len 2: 2+1 = 3\n        assert_eq!(m.row(0).iter().filter(|x| **x).count(), 9);\n        assert_eq!(m.row(1).iter().filter(|x| **x).count(), 3);\n    }\n}\n",
"src/gliner/engine.rs": "//! Model handles: an `orp::model::Model` bound to its pipeline and parameters.\n//!\n//! Expected model directory (produced by the notebook export cell):\n//! ```text\n//! <dir>/gliner_config.json\n//! <dir>/tokenizer.json            text tokenizer\n//! <dir>/model.onnx                span graph (bi-encoder: takes `labels_embeddings`)\n//! <dir>/labels_tokenizer.json     bi-encoder only\n//! <dir>/labels_encoder.onnx       bi-encoder only\n//! ```\n\nuse std::path::Path;\nuse ndarray::{concatenate, Array2, Axis};\nuse orp::model::Model;\nuse orp::params::RuntimeParameters;\n\nuse super::encode::load_tokenizer;\nuse super::labels::LabelEncoderPipeline;\nuse super::pipeline::{SpanInput, SpanPipeline};\nuse super::{GlinerConfigFile, GlinerParams, PromptKind};\nuse crate::types::SpanPrediction;\nuse crate::Result;\n\npub struct SpanModel {\n    model: Model,\n    pipeline: SpanPipeline,\n    pub params: GlinerParams,\n    pub model_id: String,\n}\n\nimpl SpanModel {\n    pub fn load(dir: impl AsRef<Path>, model_id: &str, mut params: GlinerParams, rt: RuntimeParameters) -> Result<Self> {\n        let dir = dir.as_ref();\n        let cfg = GlinerConfigFile::from_file(dir.join(\"gliner_config.json\"))?;\n        params.max_width = cfg.max_width;\n        params.max_words = params.max_words.min(cfg.max_len);\n        let pipeline = SpanPipeline::new(load_tokenizer(dir.join(\"tokenizer.json\"))?, cfg.prompt_kind(), cfg.pooling()?);\n        let model = Model::new(dir.join(\"model.onnx\"), rt)?;\n        Ok(Self { model, pipeline, params, model_id: model_id.to_string() })\n    }\n\n    pub fn is_bi_encoder(&self) -> bool {\n        matches!(self.pipeline.kind(), PromptKind::BiEncoder)\n    }\n\n    pub fn predict(&self, input: SpanInput) -> Result<Vec<Vec<SpanPrediction>>> {\n        if input.labels.is_empty() {\n            return Ok(vec![vec![]; input.texts.len()]);\n        }\n        self.model.inference(input, &self.pipeline, &self.params)\n    }\n}\n\npub struct LabelEncoder {\n    model: Model,\n    pipeline: LabelEncoderPipeline,\n    pub batch_size: usize,\n}\n\nimpl LabelEncoder {\n    pub fn load(dir: impl AsRef<Path>, max_tokens: usize, rt: RuntimeParameters) -> Result<Self> {\n        let dir = dir.as_ref();\n        let pipeline = LabelEncoderPipeline::new(load_tokenizer(dir.join(\"labels_tokenizer.json\"))?, max_tokens);\n        let model = Model::new(dir.join(\"labels_encoder.onnx\"), rt)?;\n        Ok(Self { model, pipeline, batch_size: 32 })\n    }\n\n    /// Python `encode_labels(labels, batch_size)` → (N, H).\n    pub fn encode(&self, labels: &[String]) -> Result<Array2<f32>> {\n        let mut parts = Vec::new();\n        for chunk in labels.chunks(self.batch_size.max(1)) {\n            parts.push(self.model.inference(chunk.to_vec(), &self.pipeline, &())?);\n        }\n        if parts.is_empty() {\n            return Ok(Array2::zeros((0, 0)));\n        }\n        let views: Vec<_> = parts.iter().map(|p| p.view()).collect();\n        Ok(concatenate(Axis(0), &views)?)\n    }\n}\n",
"src/gliner/labels.rs": "//! `orp` pipeline for the bi-encoder **label encoder** (Python `model.encode_labels`).\n//!\n//! Graph contract (written by the notebook's export step):\n//!   inputs  `input_ids` i64 [N, T], `attention_mask` i64 [N, T]\n//!   output  `embeddings` f32 [N, H]  (labels_encoder → labels_projection → masked mean-pool)\n//! Mean pooling is masked, so padding to the longest label equals Python's `padding=\"max_length\"`.\n\nuse composable::Composable;\nuse ndarray::Array2;\nuse orp::pipeline::{Pipeline, PostProcessor, PreProcessor};\nuse ort::session::{SessionInputs, SessionOutputs};\nuse tokenizers::Tokenizer;\nuse crate::Result;\n\npub const EMBEDDINGS: &str = \"embeddings\";\n\npub struct LabelEncoderPipeline {\n    tokenizer: Tokenizer,\n    /// Max label tokens (GLinker sets `labels_tokenizer.model_max_length = max_length`).\n    pub max_tokens: usize,\n}\n\nimpl LabelEncoderPipeline {\n    pub fn new(tokenizer: Tokenizer, max_tokens: usize) -> Self {\n        Self { tokenizer, max_tokens }\n    }\n}\n\nstruct Tokenize<'p> {\n    tok: &'p Tokenizer,\n    max_tokens: usize,\n}\nimpl<'a> Composable<Vec<String>, (SessionInputs<'a, 'a>, usize)> for Tokenize<'_> {\n    fn apply(&self, labels: Vec<String>) -> Result<(SessionInputs<'a, 'a>, usize)> {\n        let encs = self.tok.encode_batch(labels.iter().map(String::as_str).collect::<Vec<_>>(), true)?;\n        let t = encs.iter().map(|e| e.get_ids().len().min(self.max_tokens)).max().unwrap_or(1).max(1);\n        let n = encs.len();\n        let mut ids = Array2::<i64>::zeros((n, t));\n        let mut mask = Array2::<i64>::zeros((n, t));\n        for (i, e) in encs.iter().enumerate() {\n            let mut row: Vec<u32> = e.get_ids().to_vec();\n            if row.len() > self.max_tokens {\n                // keep the final special token (e.g. [SEP]) like HF truncation\n                let last = *row.last().unwrap();\n                row.truncate(self.max_tokens);\n                *row.last_mut().unwrap() = last;\n            }\n            for (j, id) in row.iter().enumerate() {\n                ids[[i, j]] = *id as i64;\n                mask[[i, j]] = 1;\n            }\n        }\n        let inputs = ort::inputs! { \"input_ids\" => ids, \"attention_mask\" => mask }?.into();\n        Ok((inputs, n))\n    }\n}\n\nstruct Extract;\nimpl<'a> Composable<(SessionOutputs<'a, 'a>, usize), Array2<f32>> for Extract {\n    fn apply(&self, (out, n): (SessionOutputs<'a, 'a>, usize)) -> Result<Array2<f32>> {\n        let v = out.get(EMBEDDINGS).ok_or(\"label encoder has no `embeddings` output\")?;\n        let a = v.try_extract_tensor::<f32>()?;\n        let a = a.into_dimensionality::<ndarray::Ix2>()?.to_owned();\n        if a.nrows() != n {\n            return Err(\"label encoder returned wrong row count\".into());\n        }\n        Ok(a)\n    }\n}\n\nimpl<'a> Pipeline<'a> for LabelEncoderPipeline {\n    type Input = Vec<String>;\n    type Output = Array2<f32>;\n    type Context = usize;\n    type Parameters = ();\n\n    fn pre_processor(&self, _: &()) -> impl PreProcessor<'a, Self::Input, Self::Context> {\n        Tokenize { tok: &self.tokenizer, max_tokens: self.max_tokens }\n    }\n    fn post_processor(&self, _: &()) -> impl PostProcessor<'a, Self::Output, Self::Context> {\n        Extract\n    }\n    fn expected_inputs(&self, _: &()) -> Option<impl Iterator<Item = &str>> {\n        Some([\"input_ids\", \"attention_mask\"].into_iter())\n    }\n    fn expected_outputs(&self, _: &()) -> Option<impl Iterator<Item = &str>> {\n        Some([EMBEDDINGS].into_iter())\n    }\n}\n",
"src/gliner/mod.rs": "//! GLiNER span-mode inference on `orp` (uni-encoder and bi-encoder), with GLinker's\n//! `input_spans` constraint and `class_probs` output — the two features `gline-rs` lacks\n//! and GLinker's L3/L4 depend on.\n\npub mod decode;\npub mod encode;\npub mod engine;\npub mod labels;\npub mod pipeline;\n\nuse serde::Deserialize;\n\n/// Inference parameters (Python `predict_entities(..., threshold, flat_ner, multi_label)`).\n#[derive(Clone, Debug)]\npub struct GlinerParams {\n    pub threshold: f32,\n    pub flat_ner: bool,\n    pub multi_label: bool,\n    /// Maximum span width in words (`gliner_config.max_width`).\n    pub max_width: usize,\n    /// Maximum words per text (`gliner_config.max_len`).\n    pub max_words: usize,\n    /// Number of `class_probs` kept per span (Python hard-codes 5).\n    pub top_k_class_probs: usize,\n}\n\nimpl Default for GlinerParams {\n    fn default() -> Self {\n        Self { threshold: 0.5, flat_ner: true, multi_label: false, max_width: 12, max_words: 384, top_k_class_probs: 5 }\n    }\n}\n\n/// How labels reach the model.\n#[derive(Clone, Debug, PartialEq)]\npub enum PromptKind {\n    /// Labels encoded separately; text graph takes `labels_embeddings` [C, H].\n    BiEncoder,\n    /// Labels are prepended to the text: `<<ENT>> l1 <<ENT>> l2 ... <<SEP>> words...`.\n    UniEncoder { ent_token: String, sep_token: String },\n}\n\n/// Subset of `gliner_config.json` needed at inference time.\n#[derive(Clone, Debug, Deserialize)]\npub struct GlinerConfigFile {\n    #[serde(default = \"d_max_width\")]\n    pub max_width: usize,\n    #[serde(default = \"d_max_len\")]\n    pub max_len: usize,\n    #[serde(default = \"d_ent\")]\n    pub ent_token: String,\n    #[serde(default = \"d_sep\")]\n    pub sep_token: String,\n    #[serde(default)]\n    pub labels_encoder: Option<String>,\n    #[serde(default)]\n    pub span_mode: Option<String>,\n    #[serde(default = \"d_pool\")]\n    pub subtoken_pooling: String,\n    #[serde(default = \"d_split\")]\n    pub words_splitter_type: String,\n}\nfn d_pool() -> String { \"first\".into() }\nfn d_split() -> String { \"whitespace\".into() }\n\n/// GLiNER `subtoken_pooling`: which sub-tokens get a word id in `words_mask`.\n#[derive(Clone, Copy, Debug, PartialEq)]\npub enum Pooling { First, Last, All }\nimpl Pooling {\n    pub fn parse(s: &str) -> crate::Result<Self> {\n        match s {\n            \"first\" => Ok(Pooling::First),\n            \"last\" => Ok(Pooling::Last),\n            \"mean\" | \"max\" => Ok(Pooling::All),\n            o => Err(format!(\"unsupported subtoken_pooling {o:?}\").into()),\n        }\n    }\n}\nfn d_max_width() -> usize { 12 }\nfn d_max_len() -> usize { 384 }\nfn d_ent() -> String { \"<<ENT>>\".into() }\nfn d_sep() -> String { \"<<SEP>>\".into() }\n\nimpl GlinerConfigFile {\n    pub fn from_file<P: AsRef<std::path::Path>>(p: P) -> crate::Result<Self> {\n        Ok(serde_json::from_str(&std::fs::read_to_string(p)?)?)\n    }\n    pub fn pooling(&self) -> crate::Result<Pooling> {\n        if self.words_splitter_type != \"whitespace\" {\n            return Err(format!(\"words_splitter_type {:?} not ported (only the default whitespace regex)\", self.words_splitter_type).into());\n        }\n        Pooling::parse(&self.subtoken_pooling)\n    }\n    pub fn prompt_kind(&self) -> PromptKind {\n        if self.labels_encoder.is_some() {\n            PromptKind::BiEncoder\n        } else {\n            PromptKind::UniEncoder { ent_token: self.ent_token.clone(), sep_token: self.sep_token.clone() }\n        }\n    }\n}\n",
"src/gliner/pipeline.rs": "//! `orp` pipeline for GLiNER span models.\n//!\n//! pre:  SpanInput ─Split─▶ Prepared ─Encode─▶ (SessionInputs, SpanContext)\n//! post: (SessionOutputs, SpanContext) ─Decode─▶ Vec<Vec<SpanPrediction>>\n\nuse std::collections::HashSet;\nuse composable::{composed, Composable};\nuse ndarray::Array2;\nuse orp::pipeline::{Pipeline, PostProcessor, PreProcessor};\nuse ort::session::{SessionInputs, SessionOutputs};\nuse tokenizers::Tokenizer;\n\nuse super::decode::{decode_item, greedy};\nuse super::encode::{encode, span_tensors, uni_prompt};\nuse super::{GlinerParams, Pooling, PromptKind};\nuse crate::text::{Word, WordSplitter};\nuse crate::types::SpanPrediction;\nuse crate::Result;\n\n/// Request for one forward pass over a batch of texts sharing one label set.\npub struct SpanInput {\n    pub texts: Vec<String>,\n    pub labels: Vec<String>,\n    /// Required for bi-encoders: (num_labels, hidden) in `labels` order.\n    pub label_embeddings: Option<Array2<f32>>,\n    /// Per text: `None` = score all spans; `Some(spans)` = only these char spans (GLinker `input_spans`).\n    pub input_spans: Vec<Option<Vec<(usize, usize)>>>,\n}\n\npub struct SpanContext {\n    pub texts: Vec<String>,\n    pub words: Vec<Vec<Word>>,\n    pub labels: Vec<String>,\n    pub filters: Vec<Option<HashSet<(usize, usize)>>>,\n}\n\nstruct Prepared {\n    input: SpanInput,\n    words: Vec<Vec<Word>>,\n    filters: Vec<Option<HashSet<(usize, usize)>>>,\n}\n\npub const BASE_INPUTS: [&str; 6] = [\"input_ids\", \"attention_mask\", \"words_mask\", \"text_lengths\", \"span_idx\", \"span_mask\"];\npub const LABELS_EMBEDDINGS: &str = \"labels_embeddings\";\npub const LOGITS: &str = \"logits\";\n\npub struct SpanPipeline {\n    tokenizer: Tokenizer,\n    splitter: WordSplitter,\n    kind: PromptKind,\n    pooling: Pooling,\n}\n\nimpl SpanPipeline {\n    pub fn new(tokenizer: Tokenizer, kind: PromptKind, pooling: Pooling) -> Self {\n        Self { tokenizer, splitter: WordSplitter::default(), kind, pooling }\n    }\n    pub fn kind(&self) -> &PromptKind {\n        &self.kind\n    }\n    pub fn splitter(&self) -> &WordSplitter {\n        &self.splitter\n    }\n}\n\n/// Char spans → word spans (Python `_convert_spans_to_word_indices`; misaligned spans dropped).\npub fn char_to_word_spans(words: &[Word], spans: &[(usize, usize)]) -> HashSet<(usize, usize)> {\n    let starts: std::collections::HashMap<usize, usize> = words.iter().enumerate().map(|(i, w)| (w.start, i)).collect();\n    let ends: std::collections::HashMap<usize, usize> = words.iter().enumerate().map(|(i, w)| (w.end, i)).collect();\n    spans\n        .iter()\n        .filter_map(|(s, e)| match (starts.get(s), ends.get(e)) {\n            (Some(&a), Some(&b)) if b >= a => Some((a, b)),\n            _ => None,\n        })\n        .collect()\n}\n\nstruct Split<'p> {\n    splitter: &'p WordSplitter,\n    max_words: usize,\n}\nimpl Composable<SpanInput, Prepared> for Split<'_> {\n    fn apply(&self, input: SpanInput) -> Result<Prepared> {\n        if input.input_spans.len() != input.texts.len() {\n            return Err(\"input_spans must have one entry per text\".into());\n        }\n        let words: Vec<Vec<Word>> = input.texts.iter().map(|t| self.splitter.split(t, self.max_words)).collect();\n        let filters = words\n            .iter()\n            .zip(&input.input_spans)\n            .map(|(w, s)| s.as_ref().map(|s| char_to_word_spans(w, s)))\n            .collect();\n        Ok(Prepared { input, words, filters })\n    }\n}\n\nstruct Encode<'p> {\n    tokenizer: &'p Tokenizer,\n    kind: &'p PromptKind,\n    pooling: Pooling,\n    max_width: usize,\n}\nimpl<'a> Composable<Prepared, (SessionInputs<'a, 'a>, SpanContext)> for Encode<'_> {\n    fn apply(&self, p: Prepared) -> Result<(SessionInputs<'a, 'a>, SpanContext)> {\n        let prompt = match self.kind {\n            PromptKind::BiEncoder => vec![],\n            PromptKind::UniEncoder { ent_token, sep_token } => uni_prompt(&p.input.labels, ent_token, sep_token),\n        };\n        let enc = encode(self.tokenizer, &prompt, &p.words, self.pooling)?;\n        let lens: Vec<usize> = p.words.iter().map(Vec::len).collect();\n        let (span_idx, span_mask) = span_tensors(&lens, enc.num_words, self.max_width);\n        let inputs: SessionInputs<'a, 'a> = match self.kind {\n            PromptKind::BiEncoder => {\n                let emb = p.input.label_embeddings.ok_or(\"bi-encoder requires label_embeddings\")?;\n                if emb.nrows() != p.input.labels.len() {\n                    return Err(\"label_embeddings rows != labels\".into());\n                }\n                ort::inputs! {\n                    \"input_ids\" => enc.input_ids,\n                    \"attention_mask\" => enc.attention_mask,\n                    \"words_mask\" => enc.words_mask,\n                    \"text_lengths\" => enc.text_lengths,\n                    \"span_idx\" => span_idx,\n                    \"span_mask\" => span_mask,\n                    LABELS_EMBEDDINGS => emb,\n                }?\n                .into()\n            }\n            PromptKind::UniEncoder { .. } => ort::inputs! {\n                \"input_ids\" => enc.input_ids,\n                \"attention_mask\" => enc.attention_mask,\n                \"words_mask\" => enc.words_mask,\n                \"text_lengths\" => enc.text_lengths,\n                \"span_idx\" => span_idx,\n                \"span_mask\" => span_mask,\n            }?\n            .into(),\n        };\n        let ctx = SpanContext { texts: p.input.texts, words: p.words, labels: p.input.labels, filters: p.filters };\n        Ok((inputs, ctx))\n    }\n}\n\nstruct Decode {\n    params: GlinerParams,\n}\nimpl<'a> Composable<(SessionOutputs<'a, 'a>, SpanContext), Vec<Vec<SpanPrediction>>> for Decode {\n    fn apply(&self, (out, ctx): (SessionOutputs<'a, 'a>, SpanContext)) -> Result<Vec<Vec<SpanPrediction>>> {\n        let logits = out.get(LOGITS).ok_or(\"model output has no `logits`\")?;\n        let arr = logits.try_extract_tensor::<f32>()?;\n        let shape = arr.shape().to_vec();\n        if shape.len() != 4 || shape[0] != ctx.texts.len() || shape[3] != ctx.labels.len() {\n            return Err(format!(\"unexpected logits shape {shape:?}\").into());\n        }\n        let mut result = Vec::with_capacity(ctx.texts.len());\n        for (b, words) in ctx.words.iter().enumerate() {\n            let item = arr.slice(ndarray::s![b, .., .., ..]);\n            let raw = decode_item(item, words.len(), ctx.filters[b].as_ref(), self.params.threshold, self.params.top_k_class_probs);\n            let kept = greedy(raw, self.params.flat_ner, self.params.multi_label);\n            let text = &ctx.texts[b];\n            result.push(\n                kept.into_iter()\n                    .map(|s| SpanPrediction {\n                        text: text[words[s.start].byte_start..words[s.end].byte_end].to_string(),\n                        label: ctx.labels[s.class].clone(),\n                        start: words[s.start].start,\n                        end: words[s.end].end,\n                        score: s.score,\n                        class_probs: s.class_probs.iter().map(|(c, p)| (ctx.labels[*c].clone(), *p)).collect(),\n                        entity_id: None,\n                    })\n                    .collect(),\n            );\n        }\n        Ok(result)\n    }\n}\n\nimpl<'a> Pipeline<'a> for SpanPipeline {\n    type Input = SpanInput;\n    type Output = Vec<Vec<SpanPrediction>>;\n    type Context = SpanContext;\n    type Parameters = GlinerParams;\n\n    fn pre_processor(&self, params: &Self::Parameters) -> impl PreProcessor<'a, Self::Input, Self::Context> {\n        composed![\n            Split { splitter: &self.splitter, max_words: params.max_words },\n            Encode { tokenizer: &self.tokenizer, kind: &self.kind, pooling: self.pooling, max_width: params.max_width }\n        ]\n    }\n\n    fn post_processor(&self, params: &Self::Parameters) -> impl PostProcessor<'a, Self::Output, Self::Context> {\n        Decode { params: params.clone() }\n    }\n\n    fn expected_inputs(&self, _params: &Self::Parameters) -> Option<impl Iterator<Item = &str>> {\n        let extra = matches!(self.kind, PromptKind::BiEncoder).then_some(LABELS_EMBEDDINGS);\n        Some(BASE_INPUTS.into_iter().chain(extra))\n    }\n\n    fn expected_outputs(&self, _params: &Self::Parameters) -> Option<impl Iterator<Item = &str>> {\n        Some([LOGITS].into_iter())\n    }\n}\n",
"src/l0.rs": "//! L0 — aggregation of L1 mentions, L2 candidates and L3/L4 links (Python `L0Component`).\n//!\n//! Divergences from Python, both deliberate:\n//! * a mention's candidates are the L2 results *for that mention* (Python re-derives them by\n//!   exact label/alias equality, which drops fuzzy hits);\n//! * the linked `entity_id` comes straight from L3's label→candidate map instead of\n//!   re-formatting templates plus a substring fallback.\n\nuse crate::template;\nuse crate::types::*;\n\n#[derive(Clone, Debug)]\npub struct L0Options {\n    pub min_confidence: f32,\n    pub include_unlinked: bool,\n    pub strict_matching: bool,\n    pub position_tolerance: usize,\n}\nimpl Default for L0Options {\n    fn default() -> Self {\n        Self { min_confidence: 0.0, include_unlinked: true, strict_matching: true, position_tolerance: 2 }\n    }\n}\n\nfn scores(p: &SpanPrediction, cands: &[DatabaseRecord], tpl: &str) -> Vec<(String, f32)> {\n    p.class_probs\n        .iter()\n        .filter_map(|(label, prob)| {\n            let l = label.trim().to_lowercase();\n            cands.iter().find(|c| template::format(tpl, c).trim().to_lowercase() == l).map(|c| (c.entity_id.clone(), *prob))\n        })\n        .collect()\n}\n\nfn linked(p: &SpanPrediction, cands: &[DatabaseRecord]) -> LinkedEntity {\n    let id = p.entity_id.clone().unwrap_or_else(|| \"unknown\".into());\n    let label = cands.iter().find(|c| c.entity_id == id).map(|c| c.label.clone()).unwrap_or_else(|| p.label.clone());\n    LinkedEntity { entity_id: id, label, confidence: p.score, start: p.start, end: p.end, matched_text: p.text.clone() }\n}\n\nfn dist(a: usize, b: usize) -> usize {\n    a.abs_diff(b)\n}\n\n/// Aggregate one text.\npub fn aggregate_text(\n    mentions: &[Mention],\n    per_mention_candidates: &[Vec<DatabaseRecord>],\n    text_candidates: &[DatabaseRecord],\n    links: &[SpanPrediction],\n    tpl: &str,\n    o: &L0Options,\n) -> Vec<L0Entity> {\n    let mut used = vec![false; links.len()];\n    let mut out = Vec::new();\n    for (mi, m) in mentions.iter().enumerate() {\n        let cands = per_mention_candidates.get(mi).cloned().unwrap_or_default();\n        let hit = links\n            .iter()\n            .position(|p| p.start == m.start && p.end == m.end)\n            .or_else(|| links.iter().position(|p| dist(p.start, m.start) <= o.position_tolerance && dist(p.end, m.end) <= o.position_tolerance));\n        let (le, cs) = match hit {\n            Some(i) => {\n                used[i] = true;\n                (Some(linked(&links[i], text_candidates)), scores(&links[i], text_candidates, tpl))\n            }\n            None => (None, vec![]),\n        };\n        let stage = if le.is_some() { Stage::L3Linked } else if !cands.is_empty() { Stage::L2Found } else { Stage::L1Only };\n        out.push(L0Entity {\n            mention_text: m.text.clone(),\n            label: m.label.clone(),\n            mention_start: m.start,\n            mention_end: m.end,\n            left_context: m.left_context.clone(),\n            right_context: m.right_context.clone(),\n            num_candidates: cands.len(),\n            candidates: cands,\n            is_linked: le.is_some(),\n            linked_entity: le,\n            candidate_scores: cs,\n            pipeline_stage: stage,\n        });\n    }\n    if !o.strict_matching {\n        for (i, p) in links.iter().enumerate() {\n            if used[i] {\n                continue;\n            }\n            let le = linked(p, text_candidates);\n            let cand: Vec<DatabaseRecord> = text_candidates.iter().filter(|c| c.entity_id == le.entity_id).take(1).cloned().collect();\n            out.push(L0Entity {\n                mention_text: p.text.clone(),\n                label: None,\n                mention_start: p.start,\n                mention_end: p.end,\n                left_context: String::new(),\n                right_context: String::new(),\n                num_candidates: cand.len(),\n                candidates: cand,\n                is_linked: true,\n                linked_entity: Some(le),\n                candidate_scores: scores(p, text_candidates, tpl),\n                pipeline_stage: Stage::L3Only,\n            });\n        }\n    }\n    // filter_by_confidence → sort_by_confidence (stable, descending)\n    out.retain(|e| match &e.linked_entity {\n        Some(l) => l.confidence >= o.min_confidence,\n        None => o.include_unlinked,\n    });\n    out.sort_by(|a, b| {\n        let ca = a.linked_entity.as_ref().map_or(0.0, |l| l.confidence);\n        let cb = b.linked_entity.as_ref().map_or(0.0, |l| l.confidence);\n        cb.total_cmp(&ca)\n    });\n    out\n}\n\npub fn stats(entities: &[Vec<L0Entity>]) -> Stats {\n    let mut s = Stats::default();\n    for e in entities.iter().flatten() {\n        s.total_mentions += 1;\n        if e.is_linked { s.linked += 1 } else { s.unlinked += 1 }\n        match e.pipeline_stage {\n            Stage::L1Only => s.l1_only += 1,\n            Stage::L2Found => s.l2_found += 1,\n            Stage::L3Linked => s.l3_linked += 1,\n            Stage::L3Only => s.l3_only += 1,\n        }\n    }\n    s.linking_rate = if s.total_mentions > 0 { s.linked as f64 / s.total_mentions as f64 } else { 0.0 };\n    s\n}\n",
"src/l1.rs": "//! L1 — mention detection.\n//!\n//! * [`L1::Gliner`]: zero-shot NER with a GLiNER span model (Python `L1GlinerComponent`),\n//!   uni- or bi-encoder; bi-encoder label embeddings are computed once at load.\n//! * [`L1::Gazetteer`]: dictionary matcher over entity labels/aliases (longest match, word\n//!   boundaries). A model-free stand-in for the spaCy L1, which has no Rust equivalent.\n//! * [`L1::Provided`]: mentions supplied by the caller (Python simple pipeline with spans).\n\nuse ndarray::Array2;\nuse crate::gliner::engine::{LabelEncoder, SpanModel};\nuse crate::gliner::pipeline::SpanInput;\nuse crate::text::{context, WordSplitter};\nuse crate::types::{DatabaseRecord, Mention};\nuse crate::Result;\n\n#[derive(Clone, Debug)]\npub struct L1Options {\n    pub max_left_context: usize,\n    pub max_right_context: usize,\n    pub min_entity_length: usize,\n}\nimpl Default for L1Options {\n    fn default() -> Self {\n        Self { max_left_context: 50, max_right_context: 50, min_entity_length: 2 }\n    }\n}\n\npub enum L1 {\n    Gliner { model: SpanModel, labels: Vec<String>, embeddings: Option<Array2<f32>>, opts: L1Options },\n    Gazetteer { terms: Vec<(Vec<String>, String)>, splitter: WordSplitter, opts: L1Options },\n    Provided { opts: L1Options },\n}\n\nimpl L1 {\n    pub fn gliner(model: SpanModel, labels: Vec<String>, encoder: Option<&LabelEncoder>, opts: L1Options) -> Result<Self> {\n        let embeddings = if model.is_bi_encoder() {\n            Some(encoder.ok_or(\"bi-encoder L1 needs a label encoder\")?.encode(&labels)?)\n        } else {\n            None\n        };\n        Ok(L1::Gliner { model, labels, embeddings, opts })\n    }\n\n    /// Build a gazetteer from entity labels + aliases (lower-cased word sequences).\n    pub fn gazetteer(entities: &[DatabaseRecord], opts: L1Options) -> Self {\n        let splitter = WordSplitter::default();\n        let mut terms: Vec<(Vec<String>, String)> = Vec::new();\n        let mut seen = std::collections::HashSet::new();\n        for e in entities {\n            for name in std::iter::once(&e.label).chain(e.aliases.iter()) {\n                let toks: Vec<String> = splitter.split(name, usize::MAX).into_iter().map(|w| w.text.to_lowercase()).collect();\n                if !toks.is_empty() && seen.insert(toks.clone()) {\n                    terms.push((toks, e.entity_type.clone()));\n                }\n            }\n        }\n        terms.sort_by(|a, b| b.0.len().cmp(&a.0.len())); // longest first\n        L1::Gazetteer { terms, splitter, opts }\n    }\n\n    fn opts(&self) -> &L1Options {\n        match self {\n            L1::Gliner { opts, .. } | L1::Gazetteer { opts, .. } | L1::Provided { opts } => opts,\n        }\n    }\n\n    pub fn extract(&self, texts: &[String]) -> Result<Vec<Vec<Mention>>> {\n        let raw: Vec<Vec<(usize, usize, Option<String>)>> = match self {\n            L1::Provided { .. } => return Err(\"L1::Provided: pass mentions to Executor::link_with_mentions\".into()),\n            L1::Gliner { model, labels, embeddings, .. } => {\n                let preds = model.predict(SpanInput {\n                    texts: texts.to_vec(),\n                    labels: labels.clone(),\n                    label_embeddings: embeddings.clone(),\n                    input_spans: vec![None; texts.len()],\n                })?;\n                preds.into_iter().map(|v| v.into_iter().map(|p| (p.start, p.end, Some(p.label))).collect()).collect()\n            }\n            L1::Gazetteer { terms, splitter, .. } => texts\n                .iter()\n                .map(|t| {\n                    let words = splitter.split(t, usize::MAX);\n                    let lw: Vec<String> = words.iter().map(|w| w.text.to_lowercase()).collect();\n                    let mut out = Vec::new();\n                    let mut i = 0;\n                    while i < words.len() {\n                        let hit = terms.iter().find(|(toks, _)| i + toks.len() <= lw.len() && lw[i..i + toks.len()] == toks[..]);\n                        if let Some((toks, ty)) = hit {\n                            let j = i + toks.len() - 1;\n                            out.push((words[i].start, words[j].end, (!ty.is_empty()).then(|| ty.clone())));\n                            i = j + 1;\n                        } else {\n                            i += 1;\n                        }\n                    }\n                    out\n                })\n                .collect(),\n        };\n        Ok(texts.iter().zip(raw).map(|(t, spans)| self.finish(t, spans)).collect())\n    }\n\n    /// Context + `filter_by_length` + dedupe + sort (Python L1 default pipeline).\n    pub fn finish(&self, text: &str, spans: Vec<(usize, usize, Option<String>)>) -> Vec<Mention> {\n        let o = self.opts();\n        let mut seen = std::collections::HashSet::new();\n        let mut v: Vec<Mention> = spans\n            .into_iter()\n            .filter(|(s, e, _)| seen.insert((*s, *e)))\n            .map(|(s, e, label)| {\n                let (l, r) = context(text, s, e, o.max_left_context, o.max_right_context);\n                Mention { text: crate::text::char_slice(text, s, e), label, start: s, end: e, left_context: l, right_context: r }\n            })\n            .filter(|m| m.text.chars().count() >= o.min_entity_length)\n            .collect();\n        v.sort_by_key(|m| m.start);\n        v\n    }\n}\n",
"src/l2.rs": "//! L2 — candidate retrieval (Python `DatabaseChainComponent` + `DictLayer`).\n//!\n//! Layers are tried by descending priority; the first layer returning anything wins.\n//! Per-mention pipeline: search → filter_by_popularity → dedupe → limit → sort_by_popularity.\n//! Redis/Elasticsearch/Postgres layers plug in through [`CandidateLayer`].\n\nuse std::collections::{HashMap, HashSet};\nuse std::io::BufRead;\nuse serde::Deserialize;\nuse crate::fuzz;\nuse crate::types::DatabaseRecord;\nuse crate::Result;\n\n#[derive(Clone, Debug, Deserialize)]\npub struct FuzzyConfig {\n    #[serde(default = \"d_min_sim\")]\n    pub min_similarity: f64,\n    #[serde(default = \"d_prefix\")]\n    pub prefix_length: usize,\n}\nfn d_min_sim() -> f64 { 0.3 }\nfn d_prefix() -> usize { 1 }\nimpl Default for FuzzyConfig {\n    fn default() -> Self { Self { min_similarity: 0.3, prefix_length: 1 } }\n}\n\n#[derive(Clone, Copy, Debug, PartialEq, Deserialize)]\n#[serde(rename_all = \"lowercase\")]\npub enum SearchMode { Exact, Fuzzy }\n\npub trait CandidateLayer: Send + Sync {\n    fn priority(&self) -> i32;\n    fn search_modes(&self) -> &[SearchMode];\n    fn search(&self, query: &str) -> Vec<DatabaseRecord>;\n    fn search_fuzzy(&self, _query: &str) -> Vec<DatabaseRecord> { vec![] }\n    fn all(&self) -> Vec<DatabaseRecord>;\n    fn update_embeddings(&mut self, ids: &[String], emb: &[Vec<f32>], model_id: &str) -> usize;\n}\n\n/// In-memory layer with label/alias indexes (Python `DictLayer`).\npub struct DictLayer {\n    pub priority: i32,\n    pub modes: Vec<SearchMode>,\n    pub fuzzy: FuzzyConfig,\n    storage: Vec<DatabaseRecord>,\n    by_id: HashMap<String, usize>,\n    label_index: HashMap<String, usize>,\n    alias_index: HashMap<String, Vec<usize>>,\n}\n\nimpl DictLayer {\n    pub fn new(priority: i32, modes: Vec<SearchMode>, fuzzy: FuzzyConfig) -> Self {\n        Self { priority, modes, fuzzy, storage: vec![], by_id: HashMap::new(), label_index: HashMap::new(), alias_index: HashMap::new() }\n    }\n\n    pub fn load_bulk(&mut self, entities: impl IntoIterator<Item = DatabaseRecord>, overwrite: bool) -> usize {\n        let mut n = 0;\n        for mut e in entities {\n            if e.source.is_empty() { e.source = \"dict\".into(); }\n            let slot = match self.by_id.get(&e.entity_id) {\n                Some(_) if !overwrite => continue,\n                Some(&i) => { self.storage[i] = e.clone(); i }\n                None => { self.storage.push(e.clone()); self.storage.len() - 1 }\n            };\n            self.by_id.insert(e.entity_id.clone(), slot);\n            self.label_index.insert(e.label.to_lowercase(), slot);\n            for a in &e.aliases {\n                let v = self.alias_index.entry(a.to_lowercase()).or_default();\n                if !v.contains(&slot) { v.push(slot); }\n            }\n            n += 1;\n        }\n        n\n    }\n\n    /// Load `entities.jsonl` (one DatabaseRecord JSON per line) or a JSON array.\n    pub fn load_file(&mut self, path: &str) -> Result<usize> {\n        let text = std::fs::read_to_string(path)?;\n        let recs: Vec<DatabaseRecord> = if text.trim_start().starts_with('[') {\n            serde_json::from_str(&text)?\n        } else {\n            let mut v = Vec::new();\n            for line in std::io::Cursor::new(text).lines() {\n                let line = line?;\n                if !line.trim().is_empty() { v.push(serde_json::from_str(&line)?); }\n            }\n            v\n        };\n        Ok(self.load_bulk(recs, true))\n    }\n\n    pub fn len(&self) -> usize { self.storage.len() }\n    pub fn is_empty(&self) -> bool { self.storage.is_empty() }\n}\n\nimpl CandidateLayer for DictLayer {\n    fn priority(&self) -> i32 { self.priority }\n    fn search_modes(&self) -> &[SearchMode] { &self.modes }\n\n    fn search(&self, query: &str) -> Vec<DatabaseRecord> {\n        let q = query.trim().to_lowercase();\n        let mut seen = HashSet::new();\n        let mut out = Vec::new();\n        if let Some(&i) = self.label_index.get(&q) {\n            seen.insert(i);\n            out.push(self.storage[i].clone());\n        }\n        if let Some(v) = self.alias_index.get(&q) {\n            for &i in v {\n                if seen.insert(i) { out.push(self.storage[i].clone()); }\n            }\n        }\n        out\n    }\n\n    fn search_fuzzy(&self, query: &str) -> Vec<DatabaseRecord> {\n        let q = query.trim().to_lowercase();\n        let prefix: String = q.chars().take(self.fuzzy.prefix_length).collect();\n        let ok_prefix = |s: &str| self.fuzzy.prefix_length == 0 || s.starts_with(&prefix);\n        let mut scored: Vec<(f64, usize)> = Vec::new();\n        for (i, e) in self.storage.iter().enumerate() {\n            let l = e.label.to_lowercase();\n            if !ok_prefix(&l) { continue; } // Python skips aliases too when the label prefix fails\n            let s = fuzz::ratio(&q, &l);\n            if s >= self.fuzzy.min_similarity {\n                scored.push((s, i));\n                continue;\n            }\n            for a in &e.aliases {\n                let a = a.to_lowercase();\n                if !ok_prefix(&a) { continue; }\n                let s = fuzz::ratio(&q, &a);\n                if s >= self.fuzzy.min_similarity { scored.push((s, i)); break; }\n            }\n        }\n        scored.sort_by(|a, b| b.0.total_cmp(&a.0));\n        scored.into_iter().map(|(_, i)| self.storage[i].clone()).collect()\n    }\n\n    fn all(&self) -> Vec<DatabaseRecord> { self.storage.clone() }\n\n    fn update_embeddings(&mut self, ids: &[String], emb: &[Vec<f32>], model_id: &str) -> usize {\n        let mut n = 0;\n        for (id, e) in ids.iter().zip(emb) {\n            if let Some(&i) = self.by_id.get(id) {\n                self.storage[i].embedding = Some(e.clone());\n                self.storage[i].embedding_model_id = Some(model_id.to_string());\n                n += 1;\n            }\n        }\n        // keep indexes pointing at the same slots; records were updated in place\n        n\n    }\n}\n\n#[derive(Clone, Debug)]\npub struct L2Options {\n    pub max_candidates: usize,\n    pub min_popularity: i64,\n}\nimpl Default for L2Options {\n    fn default() -> Self { Self { max_candidates: 30, min_popularity: 0 } }\n}\n\npub struct L2 {\n    pub layers: Vec<Box<dyn CandidateLayer>>,\n    pub opts: L2Options,\n}\n\nimpl L2 {\n    pub fn new(mut layers: Vec<Box<dyn CandidateLayer>>, opts: L2Options) -> Self {\n        layers.sort_by(|a, b| b.priority().cmp(&a.priority()));\n        Self { layers, opts }\n    }\n\n    fn dedupe(v: Vec<DatabaseRecord>) -> Vec<DatabaseRecord> {\n        let mut seen = HashSet::new();\n        v.into_iter().filter(|r| seen.insert(r.entity_id.clone())).collect()\n    }\n\n    /// Candidates for one mention string.\n    pub fn search(&self, mention: &str) -> Vec<DatabaseRecord> {\n        let mut found = Vec::new();\n        for layer in &self.layers {\n            let mut r = Vec::new();\n            for m in layer.search_modes() {\n                match m {\n                    SearchMode::Exact => r.extend(layer.search(mention)),\n                    SearchMode::Fuzzy => r.extend(layer.search_fuzzy(mention)),\n                }\n            }\n            if !r.is_empty() { found = Self::dedupe(r); break; }\n        }\n        let mut v: Vec<_> = found.into_iter().filter(|r| r.popularity >= self.opts.min_popularity).collect();\n        v = Self::dedupe(v);\n        v.truncate(self.opts.max_candidates);\n        v.sort_by(|a, b| b.popularity.cmp(&a.popularity)); // stable, like Python sorted\n        v\n    }\n\n    /// Whole database (Python `mentions=None` → simple pipeline).\n    pub fn all(&self) -> Vec<DatabaseRecord> {\n        Self::dedupe(self.layers.iter().flat_map(|l| l.all()).collect())\n    }\n\n    /// Cache embeddings into the highest-priority layer (Python `_cache_embeddings`).\n    pub fn update_embeddings(&mut self, ids: &[String], emb: &[Vec<f32>], model_id: &str) -> usize {\n        self.layers.first_mut().map(|l| l.update_embeddings(ids, emb, model_id)).unwrap_or(0)\n    }\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    fn rec(id: &str, label: &str, aliases: &[&str], pop: i64) -> DatabaseRecord {\n        DatabaseRecord { entity_id: id.into(), label: label.into(), aliases: aliases.iter().map(|s| s.to_string()).collect(), popularity: pop, ..Default::default() }\n    }\n    #[test]\n    fn exact_alias_fuzzy() {\n        let mut d = DictLayer::new(0, vec![SearchMode::Exact, SearchMode::Fuzzy], FuzzyConfig { min_similarity: 0.8, prefix_length: 1 });\n        d.load_bulk([rec(\"Q1\", \"insulin\", &[\"INS\"], 5), rec(\"Q2\", \"insulin receptor\", &[], 9), rec(\"Q3\", \"glucagon\", &[], 1)], false);\n        let l2 = L2::new(vec![Box::new(d)], L2Options::default());\n        let r = l2.search(\"Insulin\");\n        assert_eq!(r[0].entity_id, \"Q1\"); // exact first, fuzzy \"insuline\"-like extras filtered by 0.8\n        assert_eq!(l2.search(\"ins\")[0].entity_id, \"Q1\");\n        assert_eq!(l2.search(\"insuline\")[0].entity_id, \"Q1\");\n        assert!(l2.search(\"zzz\").is_empty());\n    }\n}\n",
"src/l3.rs": "//! L3 — entity linking with a GLiNER (bi-)encoder (Python `L3Processor`/`L3Component`).\n//!\n//! Candidate labels are rendered with the schema template, deduped case-insensitively,\n//! scored against L1 mention spans (`input_spans`), and each prediction carries the\n//! `entity_id` of the candidate that produced its label (no string re-matching needed).\n//! Bi-encoder label embeddings are taken from candidates when every one has an embedding\n//! for this model (`use_precomputed_embeddings`), otherwise computed by the label encoder.\n\nuse ndarray::Array2;\nuse crate::gliner::engine::{LabelEncoder, SpanModel};\nuse crate::gliner::pipeline::SpanInput;\nuse crate::template;\nuse crate::types::{DatabaseRecord, Mention, SpanPrediction};\nuse crate::Result;\n\npub struct L3 {\n    pub model: SpanModel,\n    pub encoder: Option<LabelEncoder>,\n    pub template: String,\n    pub use_precomputed_embeddings: bool,\n}\n\n/// Embeddings computed during a call, for write-back into L2 (`cache_embeddings`).\npub type NewEmbeddings = Vec<(String, Vec<f32>)>;\n\npub fn input_spans(mentions: Option<&[Vec<Mention>]>, n: usize) -> Vec<Option<Vec<(usize, usize)>>> {\n    (0..n)\n        .map(|i| {\n            mentions\n                .and_then(|m| m.get(i))\n                .filter(|v| !v.is_empty()) // Python: empty L1 list → no constraint\n                .map(|v| v.iter().map(|m| (m.start, m.end)).collect())\n        })\n        .collect()\n}\n\nimpl L3 {\n    fn embeddings_for(&self, cands: &[DatabaseRecord], idx: &[usize], labels: &[String], fresh: &mut NewEmbeddings) -> Result<Option<Array2<f32>>> {\n        if !self.model.is_bi_encoder() {\n            return Ok(None);\n        }\n        let picked: Vec<&DatabaseRecord> = idx.iter().map(|&i| &cands[i]).collect();\n        let pre = self.use_precomputed_embeddings\n            && picked.iter().all(|c| c.embedding.is_some() && c.embedding_model_id.as_deref() == Some(self.model.model_id.as_str()));\n        if pre && !picked.is_empty() {\n            let h = picked[0].embedding.as_ref().unwrap().len();\n            let mut a = Array2::<f32>::zeros((picked.len(), h));\n            for (r, c) in picked.iter().enumerate() {\n                let e = c.embedding.as_ref().unwrap();\n                if e.len() != h {\n                    return Err(format!(\"embedding dim mismatch for {}\", c.entity_id).into());\n                }\n                a.row_mut(r).assign(&ndarray::ArrayView1::from(e.as_slice()));\n            }\n            return Ok(Some(a));\n        }\n        let enc = self.encoder.as_ref().ok_or(\"bi-encoder L3 needs a label encoder\")?;\n        let a = enc.encode(labels)?;\n        for (r, c) in picked.iter().enumerate() {\n            if c.embedding.is_none() {\n                fresh.push((c.entity_id.clone(), a.row(r).to_vec()));\n            }\n        }\n        Ok(Some(a))\n    }\n\n    fn attach_ids(preds: &mut [SpanPrediction], labels: &[String], idx: &[usize], cands: &[DatabaseRecord]) {\n        for p in preds {\n            if let Some(j) = labels.iter().position(|l| *l == p.label) {\n                p.entity_id = Some(cands[idx[j]].entity_id.clone());\n            }\n        }\n    }\n\n    /// Link `texts` against per-text `candidates`. If `shared` is true all texts use\n    /// `candidates[0]` and run in one batched forward pass.\n    pub fn link(&self, texts: &[String], candidates: &[Vec<DatabaseRecord>], mentions: Option<&[Vec<Mention>]>, shared: bool) -> Result<(Vec<Vec<SpanPrediction>>, NewEmbeddings)> {\n        let spans = input_spans(mentions, texts.len());\n        let mut fresh = NewEmbeddings::new();\n        if shared && !candidates.is_empty() {\n            let cands = &candidates[0];\n            let (labels, idx) = template::labels_with_mapping(&self.template, cands);\n            let emb = self.embeddings_for(cands, &idx, &labels, &mut fresh)?;\n            let mut out = self.model.predict(SpanInput { texts: texts.to_vec(), labels: labels.clone(), label_embeddings: emb, input_spans: spans })?;\n            for v in &mut out {\n                Self::attach_ids(v, &labels, &idx, cands);\n            }\n            return Ok((out, fresh));\n        }\n        let mut out = Vec::with_capacity(texts.len());\n        for (i, text) in texts.iter().enumerate() {\n            let cands = candidates.get(i).map(Vec::as_slice).unwrap_or(&[]);\n            let (labels, idx) = template::labels_with_mapping(&self.template, cands);\n            if labels.is_empty() {\n                out.push(vec![]);\n                continue;\n            }\n            let emb = self.embeddings_for(cands, &idx, &labels, &mut fresh)?;\n            let mut v = self.model.predict(SpanInput { texts: vec![text.clone()], labels: labels.clone(), label_embeddings: emb, input_spans: vec![spans[i].clone()] })?.remove(0);\n            Self::attach_ids(&mut v, &labels, &idx, cands);\n            // Python L3 default pipeline tail: filter_by_score (already ≥ threshold) + sort_by_position\n            v.sort_by_key(|p| p.start);\n            out.push(v);\n        }\n        Ok((out, fresh))\n    }\n\n    /// Precompute embeddings for a whole entity set (Python `executor.precompute_embeddings()`).\n    pub fn precompute(&self, entities: &[DatabaseRecord]) -> Result<NewEmbeddings> {\n        let enc = match (&self.encoder, self.model.is_bi_encoder()) {\n            (Some(e), true) => e,\n            _ => return Ok(vec![]),\n        };\n        let labels: Vec<String> = entities.iter().map(|e| template::format(&self.template, e)).collect();\n        let a = enc.encode(&labels)?;\n        Ok(entities.iter().enumerate().map(|(i, e)| (e.entity_id.clone(), a.row(i).to_vec())).collect())\n    }\n}\n",
"src/l4.rs": "//! L4 — uni-encoder reranker with candidate chunking (Python `L4Processor`/`L4Component`).\n//! Labels are split into chunks of `max_labels`, each chunk scored with the L1 `input_spans`,\n//! then: deduplicate (best score per span) → filter_by_score → sort_by_position.\n\nuse crate::gliner::engine::SpanModel;\nuse crate::gliner::pipeline::SpanInput;\nuse crate::l3::input_spans;\nuse crate::template;\nuse crate::types::{DatabaseRecord, Mention, SpanPrediction};\nuse crate::Result;\n\npub struct L4 {\n    pub model: SpanModel,\n    pub template: String,\n    pub max_labels: usize,\n}\n\nimpl L4 {\n    pub fn rerank(&self, texts: &[String], candidates: &[Vec<DatabaseRecord>], mentions: Option<&[Vec<Mention>]>, shared: bool) -> Result<Vec<Vec<SpanPrediction>>> {\n        let spans = input_spans(mentions, texts.len());\n        let mut out = Vec::with_capacity(texts.len());\n        for (i, text) in texts.iter().enumerate() {\n            let cands = if shared { &candidates[0] } else { candidates.get(i).map(Vec::as_slice).unwrap_or(&[]) };\n            let (labels, idx) = template::labels_with_mapping(&self.template, cands);\n            let mut all: Vec<SpanPrediction> = Vec::new();\n            for (c, chunk) in labels.chunks(self.max_labels.max(1)).enumerate() {\n                let mut v = self\n                    .model\n                    .predict(SpanInput { texts: vec![text.clone()], labels: chunk.to_vec(), label_embeddings: None, input_spans: vec![spans[i].clone()] })?\n                    .remove(0);\n                for p in &mut v {\n                    if let Some(j) = chunk.iter().position(|l| *l == p.label) {\n                        p.entity_id = Some(cands[idx[c * self.max_labels.max(1) + j]].entity_id.clone());\n                    }\n                }\n                all.extend(v);\n            }\n            // deduplicate_entities: keep the best per (text, start, end), first-seen order\n            let mut best: Vec<SpanPrediction> = Vec::new();\n            for p in all {\n                match best.iter_mut().find(|b| b.start == p.start && b.end == p.end && b.text == p.text) {\n                    Some(b) if p.score > b.score => *b = p,\n                    Some(_) => {}\n                    None => best.push(p),\n                }\n            }\n            best.retain(|p| p.score >= self.model.params.threshold);\n            best.sort_by_key(|p| p.start);\n            out.push(best);\n        }\n        Ok(out)\n    }\n}\n",
"src/lib.rs": "//! # glinker-rs\n//!\n//! Rust port of [Knowledgator GLinker](https://github.com/Knowledgator/GLinker) built on\n//! [`orp`](https://github.com/fbilhaut/orp) (ONNX Runtime pipelines) and [`ort`](https://ort.pyke.io).\n//!\n//! Layer map (same names as the Python package):\n//!\n//! | Layer | Python                          | Rust module   |\n//! |-------|---------------------------------|---------------|\n//! | L1    | spaCy / GLiNER mention detection| [`l1`]        |\n//! | L2    | Dict/Redis/ES/PG candidate search | [`l2`] (dict layer, exact + fuzzy) |\n//! | L3    | GLiNER bi-encoder linking       | [`l3`] + [`gliner`] |\n//! | L4    | GLiNER uni-encoder reranking    | [`l4`] + [`gliner`] |\n//! | L0    | aggregation                     | [`l0`]        |\n//!\n//! Every model call goes through an `orp::pipeline::Pipeline` implementation\n//! ([`gliner::pipeline::SpanPipeline`], [`gliner::labels::LabelEncoderPipeline`]).\n\npub mod types;\npub mod text;\npub mod template;\npub mod fuzz;\npub mod gliner;\npub mod l1;\npub mod l2;\npub mod l3;\npub mod l4;\npub mod l0;\npub mod config;\npub mod executor;\npub mod metrics;\npub mod openai;\n\npub type Result<T> = orp::Result<T>;\n\npub use executor::Executor;\npub use types::*;\n",
"src/metrics.rs": "//! Matthews Correlation Coefficient for entity-linking evaluation.\n//!\n//! * [`mcc_binary`] — candidate-level: every (mention, candidate) pair is a binary decision\n//!   \"is this the linked entity?\" vs. gold. TNs exist naturally (rejected wrong candidates).\n//! * [`mcc_multiclass`] — Gorodkin R_K over mention-level labels (entity_id or `NIL`), the\n//!   multiclass generalisation used by scikit-learn's `matthews_corrcoef`.\n\nuse std::collections::{BTreeMap, BTreeSet};\nuse serde::{Deserialize, Serialize};\nuse crate::types::L0Entity;\n\npub const NIL: &str = \"NIL\";\n\n#[derive(Clone, Copy, Debug, Default, Serialize, Deserialize, PartialEq)]\npub struct Confusion {\n    pub tp: u64,\n    pub tn: u64,\n    pub fp: u64,\n    pub fn_: u64,\n}\n\nimpl Confusion {\n    pub fn add(&mut self, truth: bool, pred: bool) {\n        match (truth, pred) {\n            (true, true) => self.tp += 1,\n            (false, false) => self.tn += 1,\n            (false, true) => self.fp += 1,\n            (true, false) => self.fn_ += 1,\n        }\n    }\n}\n\n/// (TP·TN − FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN)); 0 when any marginal is 0 (sklearn convention).\npub fn mcc_binary(c: &Confusion) -> f64 {\n    let (tp, tn, fp, fn_) = (c.tp as f64, c.tn as f64, c.fp as f64, c.fn_ as f64);\n    let den = ((tp + fp) * (tp + fn_) * (tn + fp) * (tn + fn_)).sqrt();\n    if den == 0.0 { 0.0 } else { (tp * tn - fp * fn_) / den }\n}\n\n/// Gorodkin multiclass MCC over (truth, pred) label pairs.\npub fn mcc_multiclass(pairs: &[(String, String)]) -> f64 {\n    let s = pairs.len() as f64;\n    if pairs.is_empty() {\n        return 0.0;\n    }\n    let mut t: BTreeMap<&str, f64> = BTreeMap::new();\n    let mut p: BTreeMap<&str, f64> = BTreeMap::new();\n    let mut c = 0.0;\n    for (a, b) in pairs {\n        *t.entry(a).or_default() += 1.0;\n        *p.entry(b).or_default() += 1.0;\n        if a == b { c += 1.0; }\n    }\n    let keys: BTreeSet<&str> = t.keys().chain(p.keys()).copied().collect();\n    let pt: f64 = keys.iter().map(|k| p.get(k).unwrap_or(&0.0) * t.get(k).unwrap_or(&0.0)).sum();\n    let pp: f64 = p.values().map(|x| x * x).sum();\n    let tt: f64 = t.values().map(|x| x * x).sum();\n    let den = ((s * s - pp) * (s * s - tt)).sqrt();\n    if den == 0.0 { 0.0 } else { (c * s - pt) / den }\n}\n\n/// Gold annotation line: `{\"text\": \"...\", \"entities\": [{\"start\":0,\"end\":4,\"entity_id\":\"Q1\"}]}`.\n#[derive(Clone, Debug, Deserialize, Serialize)]\npub struct GoldDoc {\n    pub text: String,\n    pub entities: Vec<GoldSpan>,\n}\n#[derive(Clone, Debug, Deserialize, Serialize)]\npub struct GoldSpan {\n    pub start: usize,\n    pub end: usize,\n    pub entity_id: String,\n}\n\n#[derive(Clone, Debug, Default, Serialize)]\npub struct EvalReport {\n    pub docs: usize,\n    pub gold_mentions: usize,\n    pub predicted_links: usize,\n    pub candidate_confusion: Confusion,\n    pub mcc_candidate: f64,\n    pub mcc_mention: f64,\n    pub accuracy_mention: f64,\n}\n\n/// Score L0 output against gold. Mention items = union of gold spans and predicted spans.\npub fn evaluate(gold: &[GoldDoc], pred: &[Vec<L0Entity>]) -> EvalReport {\n    let mut r = EvalReport { docs: gold.len(), ..Default::default() };\n    let mut pairs: Vec<(String, String)> = Vec::new();\n    for (g, p) in gold.iter().zip(pred) {\n        r.gold_mentions += g.entities.len();\n        let mut gmap: BTreeMap<(usize, usize), String> = g.entities.iter().map(|e| ((e.start, e.end), e.entity_id.clone())).collect();\n        for e in p {\n            let span = (e.mention_start, e.mention_end);\n            let linked = e.linked_entity.as_ref().map(|l| l.entity_id.clone()).unwrap_or_else(|| NIL.into());\n            if linked != NIL { r.predicted_links += 1; }\n            let truth = gmap.remove(&span).unwrap_or_else(|| NIL.into());\n            // candidate-level binary decisions\n            // decision set = L2 candidates ∪ every candidate L3 scored (class_probs) ∪ the link\n            let mut ids: BTreeSet<String> = e.candidates.iter().map(|c| c.entity_id.clone()).collect();\n            ids.extend(e.candidate_scores.iter().map(|(id, _)| id.clone()));\n            if linked != NIL { ids.insert(linked.clone()); }\n            for id in ids {\n                r.candidate_confusion.add(id == truth, id == linked);\n            }\n            pairs.push((truth, linked));\n        }\n        for (_, id) in gmap {\n            // gold mention never surfaced by the pipeline: a missed link\n            r.candidate_confusion.add(true, false);\n            pairs.push((id, NIL.into()));\n        }\n    }\n    r.mcc_candidate = mcc_binary(&r.candidate_confusion);\n    r.mcc_mention = mcc_multiclass(&pairs);\n    r.accuracy_mention = if pairs.is_empty() { 0.0 } else { pairs.iter().filter(|(a, b)| a == b).count() as f64 / pairs.len() as f64 };\n    r\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    #[test]\n    fn binary_matches_sklearn() {\n        // sklearn: y_true=[1,1,1,0,0,0,0], y_pred=[1,1,0,0,0,1,0] → 0.4166666666666667\n        let c = Confusion { tp: 2, fn_: 1, fp: 1, tn: 3 };\n        assert!((mcc_binary(&c) - 0.4166666666666667).abs() < 1e-12);\n    }\n    #[test]\n    fn multiclass_matches_sklearn() {\n        // sklearn: matthews_corrcoef([0,1,2,0,1,2],[0,2,1,0,0,1]) == 0.0\n        let v: Vec<(String, String)> = [(0, 0), (1, 2), (2, 1), (0, 0), (1, 0), (2, 1)].iter().map(|(a, b)| (a.to_string(), b.to_string())).collect();\n        assert!(mcc_multiclass(&v).abs() < 1e-12);\n        // sklearn: matthews_corrcoef(list(\"aabbcca\"), list(\"abbccaa\")) == 0.34375\n        let w: Vec<(String, String)> = \"aabbcca\".chars().zip(\"abbccaa\".chars()).map(|(a, b)| (a.to_string(), b.to_string())).collect();\n        assert!((mcc_multiclass(&w) - 0.34375).abs() < 1e-12);\n        let perfect: Vec<(String, String)> = [\"a\", \"b\", \"c\"].iter().map(|x| (x.to_string(), x.to_string())).collect();\n        assert!((mcc_multiclass(&perfect) - 1.0).abs() < 1e-12);\n    }\n}\n",
"src/openai.rs": "//! OpenAI-compatible HTTP interface.\n//!\n//! * `POST /v1/chat/completions` — the last `user` message is the text to link. If its content\n//!   is JSON `{\"texts\": [...]}` every text is linked in one batch. The assistant message content\n//!   is the L0 output as JSON (`{\"entities\": [[...]], \"stats\": {...}}`). `stream: true` returns\n//!   one SSE `chat.completion.chunk` followed by `data: [DONE]`.\n//! * `GET /v1/models`, `GET /health`.\n\nuse std::sync::{Arc, Mutex};\nuse std::time::{SystemTime, UNIX_EPOCH};\nuse axum::extract::State;\nuse axum::http::{header, StatusCode};\nuse axum::response::{IntoResponse, Response};\nuse axum::routing::{get, post};\nuse axum::{Json, Router};\nuse serde::{Deserialize, Serialize};\nuse serde_json::json;\nuse crate::executor::Executor;\n\n#[derive(Clone)]\npub struct AppState {\n    pub executor: Arc<Mutex<Executor>>,\n    pub model_name: String,\n}\n\n#[derive(Debug, Deserialize)]\npub struct ChatMessage {\n    pub role: String,\n    #[serde(default)]\n    pub content: serde_json::Value,\n}\n\n#[derive(Debug, Deserialize)]\npub struct ChatRequest {\n    #[serde(default)]\n    pub model: Option<String>,\n    pub messages: Vec<ChatMessage>,\n    #[serde(default)]\n    pub stream: bool,\n}\n\n#[derive(Debug, Serialize)]\nstruct Usage {\n    prompt_tokens: usize,\n    completion_tokens: usize,\n    total_tokens: usize,\n}\n\n/// Content may be a string or OpenAI \"parts\" (`[{\"type\":\"text\",\"text\":\"...\"}]`).\nfn content_text(v: &serde_json::Value) -> String {\n    match v {\n        serde_json::Value::String(s) => s.clone(),\n        serde_json::Value::Array(parts) => parts.iter().filter_map(|p| p.get(\"text\").and_then(|t| t.as_str())).collect::<Vec<_>>().join(\"\\n\"),\n        other => other.to_string(),\n    }\n}\n\npub fn texts_from_request(req: &ChatRequest) -> Result<Vec<String>, String> {\n    let msg = req.messages.iter().rev().find(|m| m.role == \"user\").ok_or(\"no user message\")?;\n    let raw = content_text(&msg.content);\n    if let Ok(v) = serde_json::from_str::<serde_json::Value>(&raw) {\n        if let Some(arr) = v.get(\"texts\").and_then(|t| t.as_array()) {\n            return Ok(arr.iter().filter_map(|x| x.as_str().map(str::to_string)).collect());\n        }\n    }\n    Ok(vec![raw])\n}\n\nfn err(code: StatusCode, msg: impl Into<String>) -> Response {\n    (code, Json(json!({\"error\": {\"message\": msg.into(), \"type\": \"invalid_request_error\"}}))).into_response()\n}\n\nasync fn chat(State(st): State<AppState>, Json(req): Json<ChatRequest>) -> Response {\n    let texts = match texts_from_request(&req) {\n        Ok(t) => t,\n        Err(e) => return err(StatusCode::BAD_REQUEST, e),\n    };\n    let words: usize = texts.iter().map(|t| t.split_whitespace().count()).sum();\n    let ex = st.executor.clone();\n    let t2 = texts.clone();\n    let out = tokio::task::spawn_blocking(move || {\n        let mut g = ex.lock().map_err(|_| \"executor poisoned\".to_string())?;\n        g.execute(&t2).map_err(|e| e.to_string())\n    })\n    .await;\n    let out = match out {\n        Ok(Ok(o)) => o,\n        Ok(Err(e)) => return err(StatusCode::INTERNAL_SERVER_ERROR, e),\n        Err(e) => return err(StatusCode::INTERNAL_SERVER_ERROR, e.to_string()),\n    };\n    let content = serde_json::to_string(&out).unwrap_or_default();\n    let created = SystemTime::now().duration_since(UNIX_EPOCH).map(|d| d.as_secs()).unwrap_or(0);\n    let id = format!(\"chatcmpl-glinker-{created}\");\n    let model = req.model.clone().unwrap_or_else(|| st.model_name.clone());\n    let links = out.stats.linked;\n    if req.stream {\n        let chunk = json!({\n            \"id\": id, \"object\": \"chat.completion.chunk\", \"created\": created, \"model\": model,\n            \"choices\": [{\"index\": 0, \"delta\": {\"role\": \"assistant\", \"content\": content}, \"finish_reason\": \"stop\"}]\n        });\n        let body = format!(\"data: {chunk}\\n\\ndata: [DONE]\\n\\n\");\n        return ([(header::CONTENT_TYPE, \"text/event-stream\")], body).into_response();\n    }\n    Json(json!({\n        \"id\": id,\n        \"object\": \"chat.completion\",\n        \"created\": created,\n        \"model\": model,\n        \"choices\": [{\"index\": 0, \"message\": {\"role\": \"assistant\", \"content\": content}, \"finish_reason\": \"stop\"}],\n        \"usage\": Usage { prompt_tokens: words, completion_tokens: links, total_tokens: words + links },\n    }))\n    .into_response()\n}\n\nasync fn models(State(st): State<AppState>) -> Json<serde_json::Value> {\n    Json(json!({\"object\": \"list\", \"data\": [{\"id\": st.model_name, \"object\": \"model\", \"owned_by\": \"glinker-rs\"}]}))\n}\n\npub fn router(state: AppState) -> Router {\n    Router::new()\n        .route(\"/v1/chat/completions\", post(chat))\n        .route(\"/chat/completions\", post(chat))\n        .route(\"/v1/models\", get(models))\n        .route(\"/health\", get(|| async { \"ok\" }))\n        .with_state(state)\n}\n\npub async fn serve(state: AppState, addr: &str) -> std::io::Result<()> {\n    let listener = tokio::net::TcpListener::bind(addr).await?;\n    axum::serve(listener, router(state)).await\n}\n",
"src/template.rs": "//! Label templates (Python `schema.template`, e.g. `\"{label}: {description}\"`).\n\nuse crate::types::DatabaseRecord;\n\n/// Format a candidate with a `str.format`-style template.\n/// Supported fields: label, description, entity_id, entity_type, popularity, aliases.\n/// Unknown fields fall back to the plain label (Python's `KeyError` branch).\npub fn format(template: &str, r: &DatabaseRecord) -> String {\n    let mut out = String::with_capacity(template.len() + r.label.len());\n    let mut rest = template;\n    while let Some(open) = rest.find('{') {\n        out.push_str(&rest[..open]);\n        let after = &rest[open + 1..];\n        let Some(close) = after.find('}') else {\n            out.push_str(&rest[open..]);\n            return out;\n        };\n        let field = &after[..close];\n        match field {\n            \"label\" => out.push_str(&r.label),\n            \"description\" => out.push_str(&r.description),\n            \"entity_id\" => out.push_str(&r.entity_id),\n            \"entity_type\" => out.push_str(&r.entity_type),\n            \"popularity\" => out.push_str(&r.popularity.to_string()),\n            \"aliases\" => out.push_str(&format!(\"{:?}\", r.aliases)),\n            _ => return r.label.clone(),\n        }\n        rest = &after[close + 1..];\n    }\n    out.push_str(rest);\n    out\n}\n\n/// Python `_create_gliner_labels_with_mapping`: format, dedupe case-insensitively,\n/// keep first occurrence. Returns (labels, index into `candidates` per label).\npub fn labels_with_mapping(template: &str, candidates: &[DatabaseRecord]) -> (Vec<String>, Vec<usize>) {\n    let mut seen = std::collections::HashSet::new();\n    let mut labels = Vec::new();\n    let mut idx = Vec::new();\n    for (i, c) in candidates.iter().enumerate() {\n        let l = format(template, c);\n        if seen.insert(l.to_lowercase()) {\n            labels.push(l);\n            idx.push(i);\n        }\n    }\n    (labels, idx)\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    #[test]\n    fn fmt() {\n        let r = DatabaseRecord { label: \"insulin\".into(), description: \"hormone\".into(), ..Default::default() };\n        assert_eq!(format(\"{label}: {description}\", &r), \"insulin: hormone\");\n        assert_eq!(format(\"{nope}\", &r), \"insulin\");\n    }\n}\n",
"src/text.rs": "//! Word splitting and offset utilities.\n//!\n//! GLiNER's default `WhitespaceTokenSplitter` uses `\\w+(?:[-_]\\w+)*|\\S`; Rust's `regex`\n//! `\\w` is Unicode-aware like Python 3 `re`, so Tamil/Arabic text splits identically.\n\nuse regex::Regex;\n\n/// A word with byte and char offsets into the source text.\n#[derive(Clone, Debug, PartialEq)]\npub struct Word {\n    pub text: String,\n    pub byte_start: usize,\n    pub byte_end: usize,\n    pub start: usize,\n    pub end: usize,\n}\n\npub struct WordSplitter {\n    re: Regex,\n}\n\nimpl Default for WordSplitter {\n    fn default() -> Self {\n        Self { re: Regex::new(r\"\\w+(?:[-_]\\w+)*|\\S\").expect(\"static regex\") }\n    }\n}\n\nimpl WordSplitter {\n    pub fn split(&self, text: &str, max_words: usize) -> Vec<Word> {\n        let b2c = ByteToChar::new(text);\n        self.re\n            .find_iter(text)\n            .take(max_words)\n            .map(|m| Word {\n                text: m.as_str().to_string(),\n                byte_start: m.start(),\n                byte_end: m.end(),\n                start: b2c.get(m.start()),\n                end: b2c.get(m.end()),\n            })\n            .collect()\n    }\n}\n\n/// O(1) byte-offset → char-offset lookup.\npub struct ByteToChar {\n    map: Vec<usize>,\n}\n\nimpl ByteToChar {\n    pub fn new(text: &str) -> Self {\n        let mut map = vec![0usize; text.len() + 1];\n        let mut c = 0usize;\n        for (b, ch) in text.char_indices() {\n            for i in 0..ch.len_utf8() {\n                map[b + i] = c;\n            }\n            c += 1;\n        }\n        map[text.len()] = c;\n        Self { map }\n    }\n    pub fn get(&self, byte: usize) -> usize {\n        self.map[byte]\n    }\n}\n\n/// Substring by char offsets.\npub fn char_slice(text: &str, start: usize, end: usize) -> String {\n    text.chars().skip(start).take(end.saturating_sub(start)).collect()\n}\n\n/// Python L1 `_get_context`: up to `max_left`/`max_right` chars around a mention, stripped.\npub fn context(text: &str, start: usize, end: usize, max_left: usize, max_right: usize) -> (String, String) {\n    let n = text.chars().count();\n    let l = char_slice(text, start.saturating_sub(max_left), start);\n    let r = char_slice(text, end, (end + max_right).min(n));\n    (l.trim().to_string(), r.trim().to_string())\n}\n\n#[cfg(test)]\nmod tests {\n    use super::*;\n    #[test]\n    fn split_unicode() {\n        let s = WordSplitter::default();\n        let w = s.split(\"TP53-mutant தமிழ் x.\", 100);\n        let t: Vec<_> = w.iter().map(|w| w.text.as_str()).collect();\n        assert_eq!(t[0], \"TP53-mutant\");\n        assert_eq!(*t.last().unwrap(), \".\");\n        // char offsets\n        let tamil = &w[1];\n        assert_eq!(tamil.start, 12);\n        assert_eq!(char_slice(\"TP53-mutant தமிழ் x.\", tamil.start, tamil.end), tamil.text);\n    }\n}\n",
"src/types.rs": "//! Data model shared by all layers (mirrors `glinker.l*.models`).\n//! All `start`/`end` offsets are **character** (Unicode scalar) offsets, like Python `str` indices.\n\nuse serde::{Deserialize, Serialize};\n\n/// Unified entity record (Python `DatabaseRecord`).\n#[derive(Clone, Debug, Default, Serialize, Deserialize, PartialEq)]\npub struct DatabaseRecord {\n    pub entity_id: String,\n    pub label: String,\n    #[serde(default)]\n    pub aliases: Vec<String>,\n    #[serde(default)]\n    pub description: String,\n    #[serde(default)]\n    pub entity_type: String,\n    #[serde(default)]\n    pub popularity: i64,\n    #[serde(default)]\n    pub metadata: serde_json::Value,\n    #[serde(default)]\n    pub source: String,\n    #[serde(default, skip_serializing_if = \"Option::is_none\")]\n    pub embedding: Option<Vec<f32>>,\n    #[serde(default, skip_serializing_if = \"Option::is_none\")]\n    pub embedding_model_id: Option<String>,\n}\n\n/// L1 mention (Python `L1Entity`).\n#[derive(Clone, Debug, Serialize, Deserialize, PartialEq)]\npub struct Mention {\n    pub text: String,\n    #[serde(default)]\n    pub label: Option<String>,\n    pub start: usize,\n    pub end: usize,\n    #[serde(default)]\n    pub left_context: String,\n    #[serde(default)]\n    pub right_context: String,\n}\n\n/// L3/L4 span prediction (Python `L3Entity`), plus the resolved entity id.\n#[derive(Clone, Debug, Serialize, Deserialize, PartialEq)]\npub struct SpanPrediction {\n    pub text: String,\n    pub label: String,\n    pub start: usize,\n    pub end: usize,\n    pub score: f32,\n    /// Top-5 (label, probability), descending — Python `class_probs`.\n    #[serde(default)]\n    pub class_probs: Vec<(String, f32)>,\n    /// Entity id of the candidate whose formatted label produced `label` (set by L3/L4).\n    #[serde(default, skip_serializing_if = \"Option::is_none\")]\n    pub entity_id: Option<String>,\n}\n\n#[derive(Clone, Debug, Serialize, Deserialize, PartialEq)]\npub struct LinkedEntity {\n    pub entity_id: String,\n    pub label: String,\n    pub confidence: f32,\n    pub start: usize,\n    pub end: usize,\n    pub matched_text: String,\n}\n\n#[derive(Clone, Copy, Debug, Serialize, Deserialize, PartialEq, Eq)]\n#[serde(rename_all = \"snake_case\")]\npub enum Stage {\n    L1Only,\n    L2Found,\n    L3Linked,\n    L3Only,\n}\n\n/// Aggregated output (Python `L0Entity`).\n#[derive(Clone, Debug, Serialize, Deserialize, PartialEq)]\npub struct L0Entity {\n    pub mention_text: String,\n    pub label: Option<String>,\n    pub mention_start: usize,\n    pub mention_end: usize,\n    pub left_context: String,\n    pub right_context: String,\n    pub candidates: Vec<DatabaseRecord>,\n    pub num_candidates: usize,\n    pub linked_entity: Option<LinkedEntity>,\n    pub is_linked: bool,\n    /// entity_id -> probability\n    pub candidate_scores: Vec<(String, f32)>,\n    pub pipeline_stage: Stage,\n}\n\n#[derive(Clone, Debug, Default, Serialize, Deserialize, PartialEq)]\npub struct Stats {\n    pub total_mentions: usize,\n    pub linked: usize,\n    pub unlinked: usize,\n    pub linking_rate: f64,\n    pub l1_only: usize,\n    pub l2_found: usize,\n    pub l3_linked: usize,\n    pub l3_only: usize,\n}\n\n#[derive(Clone, Debug, Default, Serialize, Deserialize)]\npub struct L0Output {\n    pub entities: Vec<Vec<L0Entity>>,\n    pub stats: Stats,\n}\n"
}
CRATE = os.path.join(WORK, "glinker-rs")
for rel, body in FILES.items():
    p = pathlib.Path(CRATE, rel); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(body)
print(f"wrote {len(FILES)} files to {CRATE}")

# ── 2. export GLiNER → ONNX (custom exporter; GLiNER 0.2.30's from_labels_embeddings path ──
#      passes the flag into _build_dummy_batch and zips inputs under the wrong names)
import torch
from gliner import GLiNER

def load_gliner(model_id):
    m = GLiNER.from_pretrained(model_id).eval(); m.to("cpu")
    lt = getattr(m.data_processor, "labels_tokenizer", None)
    if lt is not None and lt.model_max_length > 100000:   # same fix GLinker's L3Component applies
        lt.model_max_length = 512
    return m

def export_gliner(model_id, out_dir):
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    m = load_gliner(model_id)
    if (out / "model.onnx").exists(): return m
    core = m.model.eval()
    probe = ["person", "organization", "country"]
    batch = m._build_dummy_batch(labels=probe)
    base = ["input_ids", "attention_mask", "words_mask", "text_lengths", "span_idx", "span_mask"]
    axes = {"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "words_mask": {0: "batch", 1: "seq"},
            "text_lengths": {0: "batch"}, "span_idx": {0: "batch", 1: "num_spans"}, "span_mask": {0: "batch", 1: "num_spans"},
            "logits": {0: "batch", 1: "words", 2: "width", 3: "classes"}}
    bi = getattr(m.config, "labels_encoder", None) is not None
    if bi:
        emb = m.encode_labels(probe).cpu()
        class TextGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask, words_mask, text_lengths, span_idx, span_mask, labels_embeddings):
                o = s.c(input_ids=input_ids, attention_mask=attention_mask, words_mask=words_mask, text_lengths=text_lengths,
                        span_idx=span_idx, span_mask=span_mask, labels_embeds=labels_embeddings)
                return o.logits if hasattr(o, "logits") else o[0]
        torch.onnx.export(TextGraph(core), tuple(batch[k] for k in base) + (emb,), str(out / "model.onnx"),
                          input_names=base + ["labels_embeddings"], output_names=["logits"],
                          dynamic_axes={**axes, "labels_embeddings": {0: "classes"}}, opset_version=19, dynamo=False)
        class LabelGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask):
                return s.c.token_rep_layer.encode_labels(input_ids, attention_mask)
        lt = m.data_processor.labels_tokenizer(probe, return_tensors="pt", padding=True)
        torch.onnx.export(LabelGraph(core), (lt["input_ids"], lt["attention_mask"]), str(out / "labels_encoder.onnx"),
                          input_names=["input_ids", "attention_mask"], output_names=["embeddings"],
                          dynamic_axes={"input_ids": {0: "n", 1: "t"}, "attention_mask": {0: "n", 1: "t"}, "embeddings": {0: "n"}},
                          opset_version=19, dynamo=False)
        tmp = out / "_labels_tok"; m.data_processor.labels_tokenizer.save_pretrained(tmp)
        shutil.copy(tmp / "tokenizer.json", out / "labels_tokenizer.json")
    else:
        class UniGraph(torch.nn.Module):
            def __init__(s, c): super().__init__(); s.c = c
            def forward(s, input_ids, attention_mask, words_mask, text_lengths, span_idx, span_mask):
                o = s.c(input_ids=input_ids, attention_mask=attention_mask, words_mask=words_mask, text_lengths=text_lengths,
                        span_idx=span_idx, span_mask=span_mask)
                return o.logits if hasattr(o, "logits") else o[0]
        torch.onnx.export(UniGraph(core), tuple(batch[k] for k in base), str(out / "model.onnx"),
                          input_names=base, output_names=["logits"], dynamic_axes=axes, opset_version=19, dynamo=False)
    m.config.to_json_file(str(out / "gliner_config.json"))
    tmp = out / "_tok"; m.data_processor.transformer_tokenizer.save_pretrained(tmp)
    shutil.copy(tmp / "tokenizer.json", out / "tokenizer.json")
    print(f"exported {model_id} → {out} ({'bi' if bi else 'uni'}-encoder)")
    return m

MODELS = os.path.join(WORK, "models")
py_linker = export_gliner(LINKER_MODEL, f"{MODELS}/linker")
if NER_MODEL: export_gliner(NER_MODEL, f"{MODELS}/ner")
if RERANK_MODEL: export_gliner(RERANK_MODEL, f"{MODELS}/rerank")

# ── 3. Lean4 formal guarantees ────────────────────────────────────────────────────────────
sh(f"cd {CRATE}/lean && lean GLinker.lean")
print("Lean4: all theorems in lean/GLinker.lean type-check ✔")

# ── 4. cargo test + release build ─────────────────────────────────────────────────────────
sh(f"cd {CRATE} && cargo test --release 2>&1 | grep -E 'test result|FAILED|panicked'", env=ENV)
sh(f"cd {CRATE} && cargo build --release", env=ENV, quiet=True)
BIN = f"{CRATE}/target/release/glinker"

# ── 5. entity DB, gold set, configs ───────────────────────────────────────────────────────
ENTS = [
    {"entity_id": "Q7240673", "label": "insulin", "aliases": ["INS"], "description": "Peptide hormone that regulates blood glucose", "entity_type": "protein", "popularity": 90},
    {"entity_id": "Q14818098", "label": "TP53", "aliases": ["p53"], "description": "Tumor suppressor gene on chromosome 17", "entity_type": "gene", "popularity": 85},
    {"entity_id": "Q17487737", "label": "BRCA1", "aliases": [], "description": "DNA repair gene linked to hereditary breast cancer", "entity_type": "gene", "popularity": 80},
    {"entity_id": "Q12206", "label": "diabetes mellitus", "aliases": ["diabetes"], "description": "Metabolic disease with high blood sugar", "entity_type": "disease", "popularity": 88},
    {"entity_id": "Q128581", "label": "breast cancer", "aliases": [], "description": "Cancer that develops from breast tissue", "entity_type": "disease", "popularity": 86},
    {"entity_id": "Q19484", "label": "metformin", "aliases": [], "description": "First-line oral medication for type 2 diabetes", "entity_type": "drug", "popularity": 70},
    {"entity_id": "Q37525", "label": "glucose", "aliases": ["blood sugar"], "description": "Simple sugar, main energy source of cells", "entity_type": "chemical", "popularity": 75},
    {"entity_id": "Q35694", "label": "jaguar", "aliases": [], "description": "Large wild cat native to the Americas", "entity_type": "animal", "popularity": 60},
    {"entity_id": "Q30055", "label": "Jaguar Cars", "aliases": ["Jaguar"], "description": "British luxury car manufacturer", "entity_type": "company", "popularity": 65},
]
GOLD_SRC = [
    ("Insulin lowers glucose in patients with diabetes.", [("Insulin", "Q7240673"), ("glucose", "Q37525"), ("diabetes", "Q12206")]),
    ("TP53 and BRCA1 mutations raise the risk of breast cancer.", [("TP53", "Q14818098"), ("BRCA1", "Q17487737"), ("breast cancer", "Q128581")]),
    ("Metformin is prescribed for diabetes mellitus.", [("Metformin", "Q19484"), ("diabetes mellitus", "Q12206")]),
    ("The jaguar hunted near the river at night.", [("jaguar", "Q35694")]),
    ("Jaguar unveiled a new electric sedan in Coventry.", [("Jaguar", "Q30055")]),
    ("Low blood sugar can follow an insulin overdose.", [("blood sugar", "Q37525"), ("insulin", "Q7240673")]),
]
def gold_doc(text, items):
    out, pos = [], 0
    for surf, eid in items:
        s = text.index(surf, pos); out.append({"start": s, "end": s + len(surf), "entity_id": eid}); pos = s + len(surf)
    return {"text": text, "entities": out}
GOLD = [gold_doc(t, it) for t, it in GOLD_SRC]
texts = [g["text"] for g in GOLD]
os.makedirs(f"{WORK}/data", exist_ok=True)
open(f"{WORK}/data/entities.jsonl", "w").write("\n".join(json.dumps(e) for e in ENTS))
open(f"{WORK}/data/gold.jsonl", "w").write("\n".join(json.dumps(g) for g in GOLD))
TEMPLATE = "{label}: {description}"
l3 = {"model_dir": "models/linker", "model_id": LINKER_MODEL, "threshold": THRESHOLD}
l1 = ({"type": "gliner", "model_dir": "models/ner", "labels": ["gene", "protein", "disease", "drug", "chemical", "animal", "company"], "threshold": 0.3}
      if NER_MODEL else {"type": "gazetteer"})
CONFIGS = {
    "simple":   {"name": "glinker-simple", "template": TEMPLATE, "entities": "data/entities.jsonl", "precompute_embeddings": True, "l1": {"type": "none"}, "l3": l3},
    "spans":    {"name": "glinker-spans", "template": TEMPLATE, "entities": "data/entities.jsonl", "precompute_embeddings": True, "l1": {"type": "gazetteer"}, "l2": {"mode": "all"}, "l3": l3},
    "pipeline": {"name": "glinker-rs", "template": TEMPLATE, "entities": "data/entities.jsonl", "precompute_embeddings": True, "l1": l1,
                 "l2": {"search_mode": ["exact", "fuzzy"], "fuzzy": {"min_similarity": 0.85}}, "l3": l3,
                 **({"l4": {"model_dir": "models/rerank", "threshold": THRESHOLD, "max_labels": 20}} if RERANK_MODEL else {})},
}
for k, c in CONFIGS.items(): json.dump(c, open(f"{WORK}/{k}.json", "w"), indent=1)

def rs_link(cfg, texts):
    return json.loads(sh(f"{BIN} link {WORK}/{cfg}.json " + " ".join(json.dumps(t) for t in texts), env=ENV, quiet=True))

# ── 6. parity: Rust (orp/ort) vs Python GLiNER on the same weights ─────────────────────────
labels = [TEMPLATE.format(**e) for e in ENTS]
emb = py_linker.encode_labels(labels)
lab2id = {l: e["entity_id"] for l, e in zip(labels, ENTS)}
def py_preds(text, spans=None):
    kw = {"input_spans": [[{"start": s, "end": e} for s, e in spans]]} if spans else {}
    r = py_linker.predict_with_embeds(text, emb, labels, threshold=THRESHOLD, flat_ner=True, multi_label=False, **kw)
    return sorted((p["start"], p["end"], lab2id[p["label"]], round(float(p["score"]), 3)) for p in r)
def rs_preds(doc):
    return sorted((x["mention_start"], x["mention_end"], x["linked_entity"]["entity_id"], round(x["linked_entity"]["confidence"], 3))
                  for x in doc if x["linked_entity"])
def compare(name, a, b, tol=2e-3):
    same = len(a) == len(b) and all(p[:3] == q[:3] and abs(p[3] - q[3]) <= tol for p, q in zip(a, b))
    print(f"  [{'MATCH' if same else 'DIFF '}] {name}\n     py: {a}\n     rs: {b}"); return same
ok = True
print("Parity A — all spans (simple pipeline):")
out = rs_link("simple", texts)
for t, d in zip(texts, out["entities"]): ok &= compare(t, py_preds(t), rs_preds(d))
print("Parity B — L1 input_spans constraint:")
out = rs_link("spans", texts)
for t, d in zip(texts, out["entities"]):
    spans = [(x["mention_start"], x["mention_end"]) for x in d]
    ok &= compare(t, py_preds(t, spans), rs_preds(d))
print("PARITY:", "PASS ✔" if ok else "DIFF — see above (scores within ±2e-3 count as equal)")

# ── 7. MCC evaluation (Rust) + sklearn cross-check ────────────────────────────────────────
from sklearn.metrics import matthews_corrcoef
rep = json.loads(sh(f"{BIN} eval {WORK}/pipeline.json {WORK}/data/gold.jsonl", env=ENV, quiet=True))
print(json.dumps(rep, indent=1))
out = rs_link("pipeline", texts)
yt, yp = [], []
for g, d in zip(GOLD, out["entities"]):
    gm = {(e["start"], e["end"]): e["entity_id"] for e in g["entities"]}
    for x in d:
        yt.append(gm.pop((x["mention_start"], x["mention_end"]), "NIL")); yp.append(x["linked_entity"]["entity_id"] if x["linked_entity"] else "NIL")
    for v in gm.values(): yt.append(v); yp.append("NIL")
sk = matthews_corrcoef(yt, yp)
print(f"MCC mention-level  rust={rep['mcc_mention']:.6f}  sklearn={sk:.6f}  {'✔ agree' if abs(sk - rep['mcc_mention']) < 1e-9 else '✘ mismatch'}")
print(f"MCC candidate-level rust={rep['mcc_candidate']:.6f}")

# ── 8. OpenAI-compatible /v1/chat/completions ─────────────────────────────────────────────
srv = subprocess.Popen([BIN, "serve", f"{WORK}/pipeline.json", f"0.0.0.0:{PORT}"], env={**os.environ, **ENV},
                       stdout=subprocess.DEVNULL, stderr=open(f"{WORK}/serve.log", "w"))
import urllib.request
for _ in range(120):
    try: urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1); break
    except Exception: time.sleep(0.5)
from openai import OpenAI
client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")
r = client.chat.completions.create(model="glinker-rs", messages=[
    {"role": "system", "content": "Link biomedical entities."},
    {"role": "user", "content": "Jaguar sponsors a diabetes charity; the jaguar is endangered."}])
res = json.loads(r.choices[0].message.content)
for e in res["entities"][0]:
    le = e["linked_entity"] or {}
    print(f"  {e['mention_text']!r:16} [{e['mention_start']}:{e['mention_end']}] → {le.get('entity_id', 'NIL'):10} {le.get('label', ''):18} p={le.get('confidence', 0):.3f}")
print("stats:", res["stats"], "| usage:", r.usage)
print(f"\nServer running (pid {srv.pid}) at http://127.0.0.1:{PORT}/v1/chat/completions — srv.terminate() to stop.")
print("Training: none — this port is inference-only. Any fine-tuning of the linker must use SAM (Sharpness-Aware Minimization) before re-export.")
